# Notebook 13: Improved STRING-GAT with active-node pooling

This notebook is a fresh Notebook 13 version of the STRING-GAT benchmark. It does not reuse Notebook 10 outputs or caches. It writes to `artefacts/reports/notebook 13`, `artefacts/cache/notebook 13`, and `artefacts/metadata/notebook 13`.

Main changes compared with Notebook 10:

- uses only connected high-confidence STRING genes;
- replaces global mean pooling with active-node masked pooling;
- treats mutation masks as mutation-present indicators instead of all-present masks;
- standardises the AUC target inside each training fold, then unscales predictions before reporting metrics;
- keeps fold tensor caching, split caching, model-state resume, prediction-file recovery, and per-row checkpointing.


In [1]:

import os
import gc
import json
import copy
import random
import hashlib
import warnings
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Iterable

import numpy as np
import pandas as pd

from IPython.display import display, Markdown

from scipy import stats
from scipy.stats import hypergeom

import matplotlib.pyplot as plt

from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, KFold, train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from torch_geometric.data import Data
    from torch_geometric.loader import DataLoader
    from torch_geometric.nn import GATv2Conv, global_add_pool
    HAS_PYG = True
except Exception as exc:
    HAS_PYG = False
    PYG_IMPORT_ERROR = exc

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 200)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

if not HAS_PYG:
    raise ImportError(
        "torch_geometric is required for Notebook 13. "
        f"Current import error: {PYG_IMPORT_ERROR}"
    )


In [2]:

# Path setup and notebook configuration

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for cand in [start] + list(start.parents):
        if (cand / "artefacts").exists() and (cand / "data").exists():
            return cand
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
ARTEFACTS = PROJECT_ROOT / "artefacts"
DATA_DIR = PROJECT_ROOT / "data"

OUT_ROOT = ARTEFACTS / "reports" / "notebook 13"
OUT_RESULTS = OUT_ROOT / "results"
OUT_FIGS = OUT_ROOT / "figures"
OUT_INTERP = OUT_ROOT / "interpretability"
OUT_COMPARE = OUT_ROOT / "baseline_comparison"
OUT_META = ARTEFACTS / "metadata" / "notebook 13"
OUT_CACHE = ARTEFACTS / "cache" / "notebook 13"

for p in [OUT_ROOT, OUT_RESULTS, OUT_FIGS, OUT_INTERP, OUT_COMPARE, OUT_META, OUT_CACHE]:
    p.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Notebook 13 output root:", OUT_ROOT)

# Same scale as Notebook 10 by default, so this run is comparable.
SEEDS = [19537, 1584678, 17052356]
PRIMARY_TARGET = "auc"
TOP_K_DRUGS = 5
N_SPLITS_DESIRED = 10
MIN_CELLS_PER_DRUG = 60
VAL_FRACTION = 0.20

ARM_THRESHOLDS = {
    "prot_procan_depmapSanger": 0.40,
    "prot_ms_ccle_gygi": 0.30,
    "prot_combined_union": 0.60,
    "prot_rppa_ccle": 0.50,
}

FEATURE_SETS = [
    "rna",
    "cnv",
    "mut",
    "prot",
    "rna+cnv",
    "rna+mut",
    "rna+prot",
    "cnv+mut",
    "cnv+prot",
    "mut+prot",
    "rna+cnv+mut",
    "rna+cnv+prot",
    "rna+mut+prot",
    "cnv+mut+prot",
    "rna+cnv+mut+prot",
]
FEATURE_SET_MODALITIES = {fs: tuple(fs.split("+")) for fs in FEATURE_SETS}

STRING_VERSION = "v12.0"
STRING_SCORE_MIN = 700

MODEL_CONFIG = {
    "hidden_channels": 32,
    "heads_first": 4,
    "dropout": 0.20,
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "max_epochs": 80,
    "patience": 12,
    "min_delta": 1e-4,
    "batch_size": 4,
    "num_workers": 0,
    "grad_clip_norm": 5.0,
    "use_edge_attr": True,
    "loss": "smooth_l1",
}

ARM_BATCH_SIZE = {
    "prot_combined_union": 3,
    "prot_procan_depmapSanger": 6,
    "prot_ms_ccle_gygi": 6,
    "prot_rppa_ccle": 6,
}

def get_batch_size_for_arm(arm_name: str) -> int:
    return int(ARM_BATCH_SIZE.get(arm_name, MODEL_CONFIG["batch_size"]))

FAST_DEV_RUN = False
FAST_DEV_LIMIT_ARMS = None
FAST_DEV_LIMIT_FEATURES = None
FAST_DEV_LIMIT_DRUGS = None
FORCE_RERUN = False

# Notebook 13 uses fresh directories and a fresh cache version.
# Resume is still enabled for interrupted Notebook 13 runs only.
RESUME_FROM_CHECKPOINT = True
USE_FOLD_TENSOR_CACHE = True
USE_SPLIT_CACHE = True
USE_MODEL_STATE_RESUME = True
WRITE_CACHE_METADATA = True
CHECKPOINT_EVERY_N_ROWS = 1
CHECKPOINT_EVERY_N_DRUGS = 5
TRAIN_STATE_SAVE_EVERY_EPOCHS = 5
CACHE_VERSION = "string_gat_notebook13_active_masked_v1"

STRING_ALIAS_PATH = DATA_DIR / "string_ppi" / "raw" / "v12.0" / "9606.protein.aliases.v12.0.txt"
STRING_LINKS_PATH = DATA_DIR / "string_ppi" / "raw" / "v12.0" / "9606.protein.links.v12.0.txt"

PATH_CANDIDATES = {
    "cell_index": [
        ARTEFACTS / "cleaned" / "notebook 2" / "cell_index.parquet",
        ARTEFACTS / "aligned" / "notebook 1" / "track2_nonintersection" / "cell_metadata.parquet",
        ARTEFACTS / "metadata" / "notebook 1" / "cell_metadata_depmap.parquet",
    ],
    "prism_long": [
        ARTEFACTS / "cleaned" / "notebook 2" / "prism_long.parquet",
    ],
    "gene_index": [
        ARTEFACTS / "cleaned" / "notebook 2" / "gene_index.parquet",
    ],
    "rna": [
        ARTEFACTS / "aligned" / "notebook 1" / "track2_nonintersection" / "rna.parquet",
        DATA_DIR / "depmap" / "processed" / "Expression_Public_25Q3_subsetted.csv",
    ],
    "cnv": [
        ARTEFACTS / "aligned" / "notebook 1" / "track2_nonintersection" / "cnv.parquet",
        DATA_DIR / "depmap" / "processed" / "Copy_Number_WGS_Public_25Q3_subsetted.csv",
    ],
    "mut": [
        ARTEFACTS / "aligned" / "notebook 1" / "track2_nonintersection" / "mut.parquet",
        DATA_DIR / "depmap" / "processed" / "Damaging_Mutations_subsetted.csv",
    ],
    "prot_procan_depmapSanger": [
        ARTEFACTS / "aligned" / "notebook 1" / "track2_nonintersection" / "prot_optional__prot_procan_depmapSanger.parquet",
    ],
    "prot_ms_ccle_gygi": [
        ARTEFACTS / "aligned" / "notebook 1" / "track2_nonintersection" / "prot_optional__prot_ms_ccle_gygi.parquet",
    ],
    "prot_combined_union": [
        ARTEFACTS / "aligned" / "notebook 1" / "track2_nonintersection" / "prot_optional__prot_combined_union.parquet",
    ],
    "prot_rppa_ccle": [
        ARTEFACTS / "aligned" / "notebook 1" / "track2_nonintersection" / "prot_optional__prot_rppa_ccle.parquet",
    ],
}

BASELINE_SCAN_DIRS = [
    ARTEFACTS / "reports" / "notebook 3b",
    ARTEFACTS / "reports" / "notebook 4",
    ARTEFACTS / "reports" / "notebook 5",
    ARTEFACTS / "reports" / "notebook 6",
    ARTEFACTS / "reports" / "notebook 7",
    ARTEFACTS / "reports" / "notebook 8",
    ARTEFACTS / "reports" / "notebook 9",
    ARTEFACTS / "reports" / "notebook 10",
    ARTEFACTS / "reports" / "notebook 11",
    ARTEFACTS / "reports" / "notebook 12",
]

PATHWAY_GMT_CANDIDATES = [
    DATA_DIR / "pathways" / "reactome_human.gmt",
    DATA_DIR / "pathways" / "Reactome_Human.gmt",
    DATA_DIR / "pathways" / "ReactomePathways.gmt",
    DATA_DIR / "pathways" / "msigdb_hallmark_human.gmt",
    DATA_DIR / "pathways" / "go_bp_human.gmt",
]

TOP10_DRUG_CANDIDATE_PATHS = [
    OUT_META / "top10_drugs.json",
    ARTEFACTS / "metadata" / "notebook 10" / "top10_drugs.json",
    ARTEFACTS / "metadata" / "notebook 11" / "top10_drugs.json",
    ARTEFACTS / "metadata" / "notebook 12" / "top10_drugs.json",
    ARTEFACTS / "reports" / "notebook 9" / "top10_drugs.json",
]

BASE_SEED = SEEDS[0]
SEED = BASE_SEED
RNG = np.random.default_rng(SEED)

def set_all_seeds(seed: int) -> None:
    global SEED, RNG
    SEED = int(seed)
    RNG = np.random.default_rng(SEED)
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    os.environ["PYTHONHASHSEED"] = str(SEED)

set_all_seeds(BASE_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

RUN_METADATA = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "device": str(DEVICE),
    "seeds": SEEDS,
    "target": PRIMARY_TARGET,
    "n_splits_desired": N_SPLITS_DESIRED,
    "min_cells_per_drug": MIN_CELLS_PER_DRUG,
    "top_k_drugs": TOP_K_DRUGS,
    "feature_sets": FEATURE_SETS,
    "arm_thresholds": ARM_THRESHOLDS,
    "string_score_min": STRING_SCORE_MIN,
    "model_config": MODEL_CONFIG,
    "cache_version": CACHE_VERSION,
    "key_changes": [
        "connected_STRING_gene_graph_only",
        "active_node_masked_pooling",
        "mutation_mask_uses_mutated_genes_only",
        "fold_safe_target_standardisation",
    ],
}
with open(OUT_META / "run_metadata.json", "w") as f:
    json.dump(RUN_METADATA, f, indent=2)


Project root: /home/andrija/Desktop/Final Year Project/FYP
Notebook 13 output root: /home/andrija/Desktop/Final Year Project/FYP/artefacts/reports/notebook 13
Using device: cuda


In [3]:

# Helper utilities

def first_existing(paths: Iterable[Path]) -> Optional[Path]:
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None

def read_table(path: Path) -> pd.DataFrame:
    path = Path(path)
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    if path.suffix in {".csv", ".txt", ".tsv"}:
        sep = "\t" if path.suffix in {".tsv", ".txt"} else ","
        if path.name.endswith("links.v12.0.txt"):
            sep = r"\s+"
        return pd.read_csv(path, sep=sep, engine="python")
    raise ValueError(f"Unsupported table format: {path}")

def sha256_file(path: Path, block_size: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(block_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def describe_file(path: Optional[Path]) -> Optional[dict]:
    if path is None:
        return None
    return {
        "path": str(path),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
        "mtime": datetime.fromtimestamp(path.stat().st_mtime, tz=timezone.utc).isoformat(),
    }

def detect_column(df: pd.DataFrame, candidates: List[str], required: bool = True) -> Optional[str]:
    lower_map = {str(c).lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    if required:
        raise KeyError(f"Could not find required column among candidates: {candidates}")
    return None

def coerce_depmap_index(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "depmap_id" in df.columns:
        df = df.set_index("depmap_id")
    elif df.index.name != "depmap_id":
        for cand in ["DepMap_ID", "DepMapID", "depmapId"]:
            if cand in df.columns:
                df = df.set_index(cand)
                break
    df.index = df.index.astype(str)
    df.index.name = "depmap_id"
    return df

def safe_numeric_frame(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in out.columns:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    return out

def to_float32_df(df: pd.DataFrame) -> pd.DataFrame:
    return df.astype(np.float32)

def canonicalise_gene_token(x: str) -> Optional[str]:
    if pd.isna(x):
        return None
    s = str(x).strip()
    if not s:
        return None
    s = s.split("::")[-1]
    for pref in ["rna__", "cnv__", "mut__", "prot__", "ms__", "rppa__", "procan__"]:
        if s.lower().startswith(pref):
            s = s[len(pref):]
    s = s.strip().upper()
    s = re.sub(r"\(.*?\)", "", s)
    s = re.split(r"[|;,/ ]", s)[0]
    s = re.split(r"[_:]", s)[0]
    s = re.sub(r"[^A-Z0-9\-]", "", s)
    if s in {"", "NA", "NAN", "NONE"}:
        return None
    if s.startswith("ENSG"):
        return None
    return s

def resolve_group_column(cell_index: pd.DataFrame) -> str:
    for cand in ["lineage_1", "primary_disease", "lineage_2", "lineage_3"]:
        if cand in cell_index.columns:
            return cand
    raise KeyError("No lineage-like grouping column found in cell_index.")

def spearman_safe(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    ok = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[ok]
    y_pred = y_pred[ok]
    if len(y_true) < 2:
        return np.nan
    if np.allclose(y_true, y_true[0]) or np.allclose(y_pred, y_pred[0]):
        return np.nan
    return float(stats.spearmanr(y_true, y_pred, nan_policy="omit").statistic)

def r2_safe(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    ok = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[ok]
    y_pred = y_pred[ok]
    if len(y_true) < 2:
        return np.nan
    try:
        return float(r2_score(y_true, y_pred))
    except Exception:
        return np.nan

def bh_fdr(pvals: np.ndarray) -> np.ndarray:
    pvals = np.asarray(pvals, dtype=float)
    n = len(pvals)
    if n == 0:
        return np.asarray([], dtype=float)
    order = np.argsort(pvals)
    ranks = np.empty(n, dtype=int)
    ranks[order] = np.arange(1, n + 1)
    adj = pvals * n / ranks
    adj_sorted = np.minimum.accumulate(adj[order][::-1])[::-1]
    out = np.empty(n, dtype=float)
    out[order] = np.clip(adj_sorted, 0.0, 1.0)
    return out

def normalise_feature_set_name(fs: str) -> str:
    return "+".join(str(fs).split("+"))

def safe_slug(x: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(x)).strip("_")

def stable_hash(obj, n: int = 16) -> str:
    payload = json.dumps(obj, sort_keys=True, ensure_ascii=False, default=str).encode("utf-8")
    return hashlib.sha1(payload).hexdigest()[:n]

def atomic_write_json(obj, path: Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.stem + ".tmp" + path.suffix)
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    os.replace(tmp, path)

def atomic_to_parquet(df: pd.DataFrame, path: Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.stem + ".tmp" + path.suffix)
    df.to_parquet(tmp, index=False)
    os.replace(tmp, path)

def atomic_to_csv(df: pd.DataFrame, path: Path, index: bool = False) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.stem + ".tmp" + path.suffix)
    df.to_csv(tmp, index=index)
    os.replace(tmp, path)

def write_cache_metadata(path: Path, meta: dict) -> None:
    if not WRITE_CACHE_METADATA:
        return
    atomic_write_json(meta, path)

def load_json_if_exists(path: Path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def update_run_state(path: Path, updates: dict) -> dict:
    state = load_json_if_exists(path, default={}) or {}
    state.update(updates)
    state["updated_at_utc"] = datetime.now(timezone.utc).isoformat()
    atomic_write_json(state, path)
    return state


In [4]:

# Data loading and harmonisation

input_registry = {}
resolved_paths = {}
for key, candidates in PATH_CANDIDATES.items():
    resolved = first_existing(candidates)
    resolved_paths[key] = resolved
    input_registry[key] = describe_file(resolved)

atomic_write_json({k: (str(v) if v is not None else None) for k, v in resolved_paths.items()}, OUT_META / "resolved_input_paths.json")

print("Resolved inputs:")
display(pd.DataFrame([{"name": k, "path": None if v is None else str(v)} for k, v in resolved_paths.items()]))

missing_required = [k for k in ["cell_index", "prism_long", "rna", "cnv", "mut"] if resolved_paths[k] is None]
if missing_required:
    raise FileNotFoundError(f"Missing required inputs: {missing_required}")

cell_index = read_table(resolved_paths["cell_index"])
if "depmap_id" not in cell_index.columns and cell_index.index.name != "depmap_id":
    cell_index = coerce_depmap_index(cell_index).reset_index()
cell_index["depmap_id"] = cell_index["depmap_id"].astype(str)
cell_index = cell_index.drop_duplicates("depmap_id").set_index("depmap_id")
group_col = resolve_group_column(cell_index)

prism_long = read_table(resolved_paths["prism_long"])
if "depmap_id" not in prism_long.columns:
    depmap_col = detect_column(prism_long, ["DepMap_ID", "depmapId", "depmap_id"])
    prism_long = prism_long.rename(columns={depmap_col: "depmap_id"})
prism_long["depmap_id"] = prism_long["depmap_id"].astype(str)

target_col = detect_column(prism_long, ["target"])
value_col = detect_column(prism_long, ["y", "value", "response"])
compound_col = detect_column(prism_long, ["compound_id", "drug_id", "compound"])
metric_col = detect_column(prism_long, ["metric"], required=False)
screen_col = detect_column(prism_long, ["screen"], required=False)

prism_long = prism_long.rename(columns={
    target_col: "target",
    value_col: "y",
    compound_col: "compound_id",
})
if metric_col and metric_col != "metric":
    prism_long = prism_long.rename(columns={metric_col: "metric"})
if screen_col and screen_col != "screen":
    prism_long = prism_long.rename(columns={screen_col: "screen"})

prism_long["compound_id"] = prism_long["compound_id"].astype(str)
prism_long["y"] = pd.to_numeric(prism_long["y"], errors="coerce")
prism_auc = prism_long[prism_long["target"].astype(str).str.lower() == PRIMARY_TARGET].copy()
prism_auc = prism_auc[prism_auc["depmap_id"].isin(cell_index.index)].copy()

def load_modality_table(key: str) -> pd.DataFrame:
    path = resolved_paths[key]
    if path is None:
        raise FileNotFoundError(f"No path resolved for {key}")
    df = read_table(path)
    df = coerce_depmap_index(df)
    df = safe_numeric_frame(df)
    return to_float32_df(df)

rna_df = load_modality_table("rna")
cnv_df = load_modality_table("cnv")
mut_df = load_modality_table("mut").fillna(0.0)

common_backbone = sorted(set(cell_index.index) & set(rna_df.index) & set(cnv_df.index) & set(mut_df.index))
cell_index = cell_index.loc[common_backbone].copy()
rna_df = rna_df.loc[common_backbone]
cnv_df = cnv_df.loc[common_backbone]
mut_df = mut_df.loc[common_backbone]
prism_auc = prism_auc[prism_auc["depmap_id"].isin(common_backbone)].copy()

print("Backbone cohort size:", len(common_backbone))
print("AUC rows in prism_long:", prism_auc.shape[0])

gene_index = None
if resolved_paths["gene_index"] is not None:
    gene_index = read_table(resolved_paths["gene_index"])
    print("Loaded gene_index:", gene_index.shape)
    print("gene_index columns:", gene_index.columns.tolist())


Resolved inputs:


,name,path
0,cell_index,/home/andrija/Desktop/Final Year Project/FYP/a...
1,prism_long,/home/andrija/Desktop/Final Year Project/FYP/a...
2,gene_index,/home/andrija/Desktop/Final Year Project/FYP/a...
3,rna,/home/andrija/Desktop/Final Year Project/FYP/a...
4,cnv,/home/andrija/Desktop/Final Year Project/FYP/a...
5,mut,/home/andrija/Desktop/Final Year Project/FYP/a...
6,prot_procan_depmapSanger,/home/andrija/Desktop/Final Year Project/FYP/a...
7,prot_ms_ccle_gygi,/home/andrija/Desktop/Final Year Project/FYP/a...
8,prot_combined_union,/home/andrija/Desktop/Final Year Project/FYP/a...
9,prot_rppa_ccle,/home/andrija/Desktop/Final Year Project/FYP/a...


Backbone cohort size: 1079
AUC rows in prism_long: 306903
Loaded gene_index: (96689, 7)
gene_index columns: ['feature_name', 'feature_base', 'platform_prefix', 'modality', 'source', 'gene_symbol_canonical', 'canonical_changed']


In [5]:

# Gene-index normalisation, feature-to-gene mapping, and top drugs

def build_gene_lookup(gene_index: Optional[pd.DataFrame]) -> Dict[str, str]:
    if gene_index is None or gene_index.empty:
        print("[WARN] gene_index is missing or empty.")
        return {}

    gi = gene_index.copy()
    cols = {str(c).lower(): c for c in gi.columns}
    gene_col = cols.get("gene_symbol_canonical")
    if gene_col is None:
        print("[WARN] Could not build gene lookup from gene_index.")
        print("Available gene_index columns:", gi.columns.tolist())
        return {}

    key_cols = []
    for cand in ["feature_name", "feature_base"]:
        col = cols.get(cand)
        if col is not None:
            key_cols.append(col)

    if not key_cols:
        print("[WARN] No usable feature key columns found in gene_index.")
        print("Available gene_index columns:", gi.columns.tolist())
        return {}

    gi[gene_col] = gi[gene_col].map(canonicalise_gene_token)
    gi = gi.dropna(subset=[gene_col]).copy()

    lookup = {}
    for key_col in key_cols:
        tmp = gi[[key_col, gene_col]].copy()
        tmp[key_col] = tmp[key_col].astype(str).str.strip()
        tmp = tmp[tmp[key_col] != ""].drop_duplicates(subset=[key_col], keep="first")
        lookup.update(dict(zip(tmp[key_col], tmp[gene_col])))

    print(f"Built gene lookup with {len(lookup)} entries using {key_cols} -> {gene_col}")
    return lookup

GENE_LOOKUP = build_gene_lookup(gene_index)
print("Gene lookup entries:", len(GENE_LOOKUP))

def feature_to_gene_map(columns: Iterable[str]) -> Dict[str, str]:
    mapping = {}
    for col in columns:
        col = str(col)
        gene = GENE_LOOKUP.get(col)
        if gene is None:
            gene = canonicalise_gene_token(col)
        if gene is not None:
            mapping[col] = gene
    return mapping

def aggregate_features_to_genes(df: pd.DataFrame, mapping: Dict[str, str]) -> pd.DataFrame:
    keep_cols = [c for c in df.columns if c in mapping]
    if not keep_cols:
        return pd.DataFrame(index=df.index)
    tmp = df[keep_cols].rename(columns=mapping)
    out = tmp.T.groupby(level=0).mean().T
    out.index = df.index.astype(str)
    out.index.name = "depmap_id"
    return to_float32_df(out)

def try_load_top10_from_json(path: Path) -> Optional[List[str]]:
    if not path.exists():
        return None
    with open(path, "r") as f:
        obj = json.load(f)
    if isinstance(obj, dict):
        for key in ["top10_drugs", "compound_ids", "drugs", "top_drugs"]:
            if key in obj and isinstance(obj[key], list) and len(obj[key]) >= TOP_K_DRUGS:
                return [str(x) for x in obj[key][:TOP_K_DRUGS]]
    if isinstance(obj, list) and len(obj) >= TOP_K_DRUGS:
        return [str(x) for x in obj[:TOP_K_DRUGS]]
    return None

resolved_top10 = None
for p in TOP10_DRUG_CANDIDATE_PATHS:
    maybe = try_load_top10_from_json(p)
    if maybe is not None:
        resolved_top10 = maybe
        print("Loaded fixed top drug list from:", p)
        break

if resolved_top10 is None:
    coverage = prism_auc.groupby("compound_id")["depmap_id"].nunique().sort_values(ascending=False)
    resolved_top10 = coverage.head(TOP_K_DRUGS).index.astype(str).tolist()
    print("Derived top drugs from AUC coverage in prism_long.")

TOP10_DRUGS = resolved_top10[:TOP_K_DRUGS]
atomic_write_json({"top10_drugs": TOP10_DRUGS}, OUT_META / "top10_drugs.json")
display(pd.DataFrame({"compound_id": TOP10_DRUGS}))


Built gene lookup with 40190 entries using ['feature_name', 'feature_base'] -> gene_symbol_canonical
Gene lookup entries: 40190
Loaded fixed top drug list from: /home/andrija/Desktop/Final Year Project/FYP/artefacts/metadata/notebook 10/top10_drugs.json


,compound_id
0,IXAZOMIB (BRD:BRD-K78659596-001-03-9)
1,OTS167 (BRD:BRD-K53417444-003-03-1)
2,SB-2343 (BRD:BRD-K98795921-001-01-7)
3,PF-05212384 (BRD:BRD-K07955840-001-02-3)
4,CR8-(R) (BRD:BRD-K40331046-305-01-5)


In [6]:

# Load and threshold proteomics arms

def load_and_prepare_proteomics_arm(arm_name: str, threshold: float) -> dict:
    path = resolved_paths[arm_name]
    if path is None:
        raise FileNotFoundError(f"Missing proteomics path for {arm_name}")

    raw = load_modality_table(arm_name)
    raw = raw.loc[raw.index.intersection(common_backbone)].copy()

    arm_observed_mask = raw.notna().any(axis=1)
    raw_arm = raw.loc[arm_observed_mask].copy()

    if raw_arm.empty:
        feature_report = pd.DataFrame(columns=["raw_feature", "kept_by_threshold", "missing_fraction_arm_observed", "mapped_gene"])
        feature_report.to_csv(OUT_META / f"{arm_name}__feature_threshold_report.csv", index=False)
        return {
            "raw": raw,
            "raw_arm": raw_arm,
            "raw_kept": pd.DataFrame(index=raw.index),
            "gene_df": pd.DataFrame(index=raw.index),
            "eligible_cells": [],
            "meta": {
                "arm": arm_name,
                "threshold": threshold,
                "n_cells_raw": int(raw.shape[0]),
                "n_cells_arm_observed": 0,
                "n_features_raw": int(raw.shape[1]),
                "n_features_kept": 0,
                "n_genes_after_aggregation": 0,
                "n_eligible_cells": 0,
                "cell_coverage_fraction": 0.0,
                "overall_missing_after_threshold": np.nan,
            },
        }

    missing_frac = raw_arm.isna().mean(axis=0)
    keep = missing_frac <= threshold

    raw_kept = raw.loc[:, keep].copy()
    raw_kept_arm = raw_arm.loc[:, keep].copy()

    mapping = feature_to_gene_map(raw_kept.columns)
    gene_df = aggregate_features_to_genes(raw_kept, mapping)

    eligible_cells = sorted(gene_df.index[gene_df.notna().any(axis=1)].astype(str).tolist())

    feature_report = pd.DataFrame({
        "raw_feature": raw.columns.astype(str),
        "kept_by_threshold": raw.columns.isin(raw_kept.columns),
        "missing_fraction_arm_observed": raw_arm.isna().mean(axis=0).reindex(raw.columns).values,
        "mapped_gene": [mapping.get(c) for c in raw.columns.astype(str)],
    })
    feature_report.to_csv(OUT_META / f"{arm_name}__feature_threshold_report.csv", index=False)

    meta = {
        "arm": arm_name,
        "threshold": threshold,
        "n_cells_raw": int(raw.shape[0]),
        "n_cells_arm_observed": int(raw_arm.shape[0]),
        "n_features_raw": int(raw.shape[1]),
        "n_features_kept": int(raw_kept.shape[1]),
        "n_genes_after_aggregation": int(gene_df.shape[1]),
        "n_eligible_cells": int(len(eligible_cells)),
        "cell_coverage_fraction": float(len(eligible_cells) / max(len(common_backbone), 1)),
        "overall_missing_after_threshold": float(raw_kept_arm.isna().mean().mean()) if raw_kept_arm.shape[1] else np.nan,
    }

    return {
        "raw": raw,
        "raw_arm": raw_arm,
        "raw_kept": raw_kept,
        "gene_df": gene_df,
        "eligible_cells": eligible_cells,
        "meta": meta,
    }

prot_arm_data = {}
for arm_name, thr in ARM_THRESHOLDS.items():
    prot_arm_data[arm_name] = load_and_prepare_proteomics_arm(arm_name, thr)

for arm_name in ARM_THRESHOLDS:
    arm_obj = prot_arm_data[arm_name]
    print(
        arm_name,
        "| raw:", arm_obj["raw"].shape,
        "| raw_kept:", arm_obj["raw_kept"].shape,
        "| gene_df:", arm_obj["gene_df"].shape,
        "| eligible_cells:", len(arm_obj["eligible_cells"]),
    )

arm_meta_df = pd.DataFrame([v["meta"] for v in prot_arm_data.values()]).sort_values("arm").reset_index(drop=True)
arm_meta_df.to_csv(OUT_META / "proteomics_arm_threshold_metadata.csv", index=False)
display(arm_meta_df)

rna_gene_pre = aggregate_features_to_genes(rna_df, feature_to_gene_map(rna_df.columns)).loc[common_backbone]
cnv_gene_pre = aggregate_features_to_genes(cnv_df, feature_to_gene_map(cnv_df.columns)).loc[common_backbone]
mut_gene_pre = aggregate_features_to_genes(mut_df, feature_to_gene_map(mut_df.columns)).loc[common_backbone]


prot_procan_depmapSanger | raw: (1079, 7906) | raw_kept: (1079, 4647) | gene_df: (1079, 4647) | eligible_cells: 485
prot_ms_ccle_gygi | raw: (1079, 11780) | raw_kept: (1079, 7880) | gene_df: (1079, 7880) | eligible_cells: 304
prot_combined_union | raw: (1079, 18751) | raw_kept: (1079, 11524) | gene_df: (1079, 7110) | eligible_cells: 679
prot_rppa_ccle | raw: (1079, 144) | raw_kept: (1079, 144) | gene_df: (1079, 144) | eligible_cells: 612


,arm,threshold,n_cells_raw,n_cells_arm_observed,n_features_raw,n_features_kept,n_genes_after_aggregation,n_eligible_cells,cell_coverage_fraction,overall_missing_after_threshold
0,prot_combined_union,0.6,1079,679,18751,11524,7110,679,0.629286,0.471390
1,prot_ms_ccle_gygi,0.3,1079,304,11780,7880,7880,304,0.281742,0.041813
2,prot_procan_depmapSanger,0.4,1079,485,7906,4647,4647,485,0.449490,0.100472
3,prot_rppa_ccle,0.5,1079,612,144,144,144,612,0.567192,0.000000


In [7]:

# Build connected high-confidence HGNC STRING graph

if not STRING_ALIAS_PATH.exists():
    raise FileNotFoundError(f"Missing STRING alias file: {STRING_ALIAS_PATH}")
if not STRING_LINKS_PATH.exists():
    raise FileNotFoundError(f"Missing STRING links file: {STRING_LINKS_PATH}")

def collect_candidate_genes() -> set:
    genes = set(rna_gene_pre.columns) | set(cnv_gene_pre.columns) | set(mut_gene_pre.columns)
    for arm_name in ARM_THRESHOLDS:
        genes |= set(prot_arm_data[arm_name]["gene_df"].columns)
    return {g for g in genes if g is not None}

candidate_genes = collect_candidate_genes()
print("Candidate HGNC genes from modalities:", len(candidate_genes))

aliases = pd.read_csv(
    STRING_ALIAS_PATH,
    sep="\t",
    usecols=["#string_protein_id", "alias", "source"],
    dtype={"#string_protein_id": "string", "alias": "string", "source": "string"},
)
aliases = aliases.rename(columns={"#string_protein_id": "string_protein_id"})
aliases["alias_norm"] = aliases["alias"].map(canonicalise_gene_token)
aliases = aliases[aliases["alias_norm"].isin(candidate_genes)].copy()

SOURCE_PRIORITY = [
    "Ensembl_HGNC_HGNC_ID",
    "Ensembl_HGNC_symbol",
    "BLAST_UniProt_GN",
    "UniProt_GN",
    "HGNC",
]

def source_rank(src: str) -> int:
    s = str(src)
    for i, key in enumerate(SOURCE_PRIORITY):
        if key.lower() in s.lower():
            return i
    return len(SOURCE_PRIORITY)

aliases["source_rank"] = aliases["source"].map(source_rank)
aliases["alias_len"] = aliases["alias_norm"].astype(str).str.len()
aliases = aliases.sort_values(["string_protein_id", "source_rank", "alias_len", "alias_norm"])

protein_to_gene = aliases.groupby("string_protein_id", as_index=True)["alias_norm"].first().to_dict()

links = pd.read_csv(
    STRING_LINKS_PATH,
    sep=r"\s+",
    engine="c",
    usecols=["protein1", "protein2", "combined_score"],
    dtype={"protein1": "string", "protein2": "string", "combined_score": np.int32},
)
links = links[links["combined_score"] >= STRING_SCORE_MIN].copy()
links["gene1"] = links["protein1"].map(protein_to_gene)
links["gene2"] = links["protein2"].map(protein_to_gene)
links = links.dropna(subset=["gene1", "gene2"]).copy()
links = links[links["gene1"] != links["gene2"]].copy()

g1 = links["gene1"].to_numpy()
g2 = links["gene2"].to_numpy()
gmin = np.where(g1 <= g2, g1, g2)
gmax = np.where(g1 <= g2, g2, g1)

tmp_edges = pd.DataFrame({
    "gene1": gmin,
    "gene2": gmax,
    "combined_score": links["combined_score"].to_numpy(),
})

gene_edges_all = tmp_edges.groupby(["gene1", "gene2"], as_index=False)["combined_score"].max()
connected_genes = set(gene_edges_all["gene1"]).union(set(gene_edges_all["gene2"]))

# Main Notebook 13 graph reduction: only candidate genes that participate in at least one retained STRING edge.
graph_nodes = sorted(set(candidate_genes) & connected_genes)
node_to_idx = {g: i for i, g in enumerate(graph_nodes)}

gene_edges = gene_edges_all[
    gene_edges_all["gene1"].isin(node_to_idx) & gene_edges_all["gene2"].isin(node_to_idx)
].copy()

edge_src_s = gene_edges["gene1"].map(node_to_idx)
edge_dst_s = gene_edges["gene2"].map(node_to_idx)
keep = edge_src_s.notna() & edge_dst_s.notna()
edge_src = edge_src_s.loc[keep].astype(np.int64).to_numpy()
edge_dst = edge_dst_s.loc[keep].astype(np.int64).to_numpy()
edge_score = gene_edges.loc[keep, "combined_score"].to_numpy(dtype=np.float32) / 1000.0

row_idx = np.concatenate([edge_src, edge_dst])
col_idx = np.concatenate([edge_dst, edge_src])
edge_w = np.concatenate([edge_score, edge_score])

edge_index = torch.tensor(np.vstack([row_idx, col_idx]), dtype=torch.long)
edge_attr = torch.tensor(edge_w.reshape(-1, 1), dtype=torch.float32)

genes_in_component = set(gene_edges["gene1"]).union(set(gene_edges["gene2"]))

node_meta = pd.DataFrame({
    "gene_symbol": graph_nodes,
    "node_idx": np.arange(len(graph_nodes), dtype=int),
    "in_string_high_confidence_component": [g in genes_in_component for g in graph_nodes],
})
node_meta.to_csv(OUT_META / "string_graph_nodes.csv", index=False)
gene_edges.to_csv(OUT_META / "string_graph_edges_gene_level.csv", index=False)

graph_meta = {
    "string_version": STRING_VERSION,
    "combined_score_min": STRING_SCORE_MIN,
    "n_candidate_genes_from_modalities": int(len(candidate_genes)),
    "n_connected_candidate_genes_retained": int(len(graph_nodes)),
    "n_candidate_genes_removed_as_disconnected": int(len(candidate_genes) - len(graph_nodes)),
    "n_edges_undirected": int(gene_edges.shape[0]),
    "n_edges_directed_for_pyg": int(edge_index.shape[1]),
    "alias_rows_retained": int(aliases.shape[0]),
    "protein_to_gene_mappings": int(len(protein_to_gene)),
    "alias_path": str(STRING_ALIAS_PATH),
    "links_path": str(STRING_LINKS_PATH),
    "graph_reduction": "candidate_genes_intersected_with_connected_STRING_genes",
}
atomic_write_json(graph_meta, OUT_META / "string_graph_metadata.json")
display(pd.DataFrame([graph_meta]))


Candidate HGNC genes from modalities: 21392


,string_version,combined_score_min,n_candidate_genes_from_modalities,n_connected_candidate_genes_retained,n_candidate_genes_removed_as_disconnected,n_edges_undirected,n_edges_directed_for_pyg,alias_rows_retained,protein_to_gene_mappings,alias_path,links_path,graph_reduction
0,v12.0,700,21392,15893,5499,232301,464602,170007,19178,/home/andrija/Desktop/Final Year Project/FYP/d...,/home/andrija/Desktop/Final Year Project/FYP/d...,candidate_genes_intersected_with_connected_STR...


In [8]:

# Build graph-aligned modality matrices

rna_gene = rna_gene_pre.copy()
cnv_gene = cnv_gene_pre.copy()
mut_gene = mut_gene_pre.copy()

def reindex_genes_to_graph(df: pd.DataFrame) -> pd.DataFrame:
    out = df.reindex(index=common_backbone).reindex(columns=graph_nodes)
    out.index.name = "depmap_id"
    return to_float32_df(out)

rna_gene = reindex_genes_to_graph(rna_gene)
cnv_gene = reindex_genes_to_graph(cnv_gene)
mut_gene = reindex_genes_to_graph(mut_gene)

prot_gene_by_arm = {}
for arm_name in ARM_THRESHOLDS:
    prot_gene_by_arm[arm_name] = (
        prot_arm_data[arm_name]["gene_df"]
        .reindex(index=common_backbone)
        .reindex(columns=graph_nodes)
        .astype(np.float32)
    )

modality_coverage_report = []
for name, df in [("rna", rna_gene), ("cnv", cnv_gene), ("mut", mut_gene)]:
    modality_coverage_report.append({
        "modality": name,
        "n_cells": int(df.shape[0]),
        "n_nodes": int(df.shape[1]),
        "overall_missing": float(df.isna().mean().mean()),
        "nonmissing_gene_fraction": float((df.notna().sum(axis=0) > 0).mean()),
    })
for arm_name, df in prot_gene_by_arm.items():
    modality_coverage_report.append({
        "modality": arm_name,
        "n_cells": int(df.shape[0]),
        "n_nodes": int(df.shape[1]),
        "overall_missing": float(df.isna().mean().mean()),
        "nonmissing_gene_fraction": float((df.notna().sum(axis=0) > 0).mean()),
    })

coverage_df = pd.DataFrame(modality_coverage_report)
coverage_df.to_csv(OUT_META / "graph_aligned_modality_coverage.csv", index=False)
display(coverage_df)


,modality,n_cells,n_nodes,overall_missing,nonmissing_gene_fraction
0,rna,1079,15893,0.007865,0.992135
1,cnv,1079,15893,0.030580,0.969420
2,mut,1079,15893,0.071038,0.928962
3,prot_procan_depmapSanger,1079,15893,0.887525,0.277229
4,prot_ms_ccle_gygi,1079,15893,0.874470,0.463789
5,prot_combined_union,1079,15893,0.828633,0.412320
6,prot_rppa_ccle,1079,15893,1.000000,0.000000


In [9]:

# Split helpers and fold-safe scaling

def safe_group_folds(sample_ids: List[str], groups: pd.Series, seed: int, n_splits_desired: int = N_SPLITS_DESIRED):
    sample_ids = np.asarray(sample_ids, dtype=object)
    groups = groups.loc[sample_ids].astype(str)
    perm_rng = np.random.default_rng(seed)
    perm = perm_rng.permutation(len(sample_ids))
    sample_ids = sample_ids[perm]
    groups = groups.iloc[perm]

    unique_groups = groups.nunique(dropna=False)
    if unique_groups >= 2:
        n_splits = max(2, min(n_splits_desired, unique_groups))
        splitter = GroupKFold(n_splits=n_splits)
        folds = []
        for tr, te in splitter.split(sample_ids, groups=groups):
            folds.append((sample_ids[tr].tolist(), sample_ids[te].tolist()))
        return folds, f"GroupKFold({n_splits})"

    n_splits = max(2, min(n_splits_desired, len(sample_ids)))
    splitter = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    folds = []
    for tr, te in splitter.split(sample_ids):
        folds.append((sample_ids[tr].tolist(), sample_ids[te].tolist()))
    return folds, f"KFold({n_splits})"

def preview_outer_splitter(groups: pd.Series, n_obs: int) -> str:
    unique_groups = groups.astype(str).nunique(dropna=False)
    if unique_groups >= 2:
        n_splits = max(2, min(N_SPLITS_DESIRED, unique_groups))
        return f"GroupKFold(n_splits={n_splits})"
    n_splits = max(2, min(N_SPLITS_DESIRED, n_obs))
    return f"KFold(n_splits={n_splits})"

def get_outer_folds_cached(sample_ids: List[str], groups: pd.Series, seed: int, arm_name: str, compound_id: str):
    split_dir = OUT_CACHE / "splits"
    split_dir.mkdir(parents=True, exist_ok=True)
    cache_payload = {
        "cache_version": CACHE_VERSION,
        "seed": int(seed),
        "arm": arm_name,
        "compound_id": str(compound_id),
        "sample_ids": list(map(str, sample_ids)),
        "groups": groups.loc[sample_ids].astype(str).to_dict(),
        "n_splits_desired": int(N_SPLITS_DESIRED),
    }
    cache_key = stable_hash(cache_payload, n=24)
    cache_path = split_dir / f"outer_folds__{cache_key}.json"
    if USE_SPLIT_CACHE and cache_path.exists() and not FORCE_RERUN:
        cached = load_json_if_exists(cache_path, default={}) or {}
        folds = [(x[0], x[1]) for x in cached.get("folds", [])]
        splitter_name = cached.get("splitter_name")
        if folds and splitter_name is not None:
            return folds, splitter_name, cache_path

    folds, splitter_name = safe_group_folds(sample_ids, groups, seed, N_SPLITS_DESIRED)
    atomic_write_json({**cache_payload, "splitter_name": splitter_name, "folds": folds}, cache_path)
    return folds, splitter_name, cache_path

def make_train_val_split(train_ids: List[str], groups: pd.Series, seed: int, val_fraction: float = VAL_FRACTION):
    train_ids = np.asarray(train_ids, dtype=object)
    grp = groups.loc[train_ids].astype(str)
    if len(train_ids) < 8:
        n_val = max(1, int(round(len(train_ids) * val_fraction)))
        perm = np.random.default_rng(seed).permutation(len(train_ids))
        val_idx = perm[:n_val]
        tr_idx = perm[n_val:]
        return train_ids[tr_idx].tolist(), train_ids[val_idx].tolist(), "random_small"

    if grp.nunique() >= 2:
        try:
            gss = GroupShuffleSplit(n_splits=1, test_size=val_fraction, random_state=seed)
            tr_idx, va_idx = next(gss.split(train_ids, groups=grp))
            return train_ids[tr_idx].tolist(), train_ids[va_idx].tolist(), "GroupShuffleSplit"
        except Exception:
            pass

    tr_ids, va_ids = train_test_split(train_ids, test_size=val_fraction, random_state=seed, shuffle=True)
    return list(tr_ids), list(va_ids), "train_test_split"

def get_train_val_split_cached(train_ids: List[str], groups: pd.Series, seed: int, arm_name: str, compound_id: str, fold_idx: int):
    split_dir = OUT_CACHE / "splits"
    split_dir.mkdir(parents=True, exist_ok=True)
    cache_payload = {
        "cache_version": CACHE_VERSION,
        "seed": int(seed),
        "arm": arm_name,
        "compound_id": str(compound_id),
        "fold_idx": int(fold_idx),
        "train_ids": list(map(str, train_ids)),
        "groups": groups.loc[train_ids].astype(str).to_dict(),
        "val_fraction": float(VAL_FRACTION),
    }
    cache_key = stable_hash(cache_payload, n=24)
    cache_path = split_dir / f"train_val_split__{cache_key}.json"
    if USE_SPLIT_CACHE and cache_path.exists() and not FORCE_RERUN:
        cached = load_json_if_exists(cache_path, default={}) or {}
        inner_train_ids = cached.get("inner_train_ids")
        val_ids = cached.get("val_ids")
        splitter_name = cached.get("splitter_name")
        if inner_train_ids and val_ids and splitter_name is not None:
            return inner_train_ids, val_ids, splitter_name, cache_path

    inner_train_ids, val_ids, splitter_name = make_train_val_split(train_ids, groups, seed, VAL_FRACTION)
    atomic_write_json({
        **cache_payload,
        "splitter_name": splitter_name,
        "inner_train_ids": inner_train_ids,
        "val_ids": val_ids,
    }, cache_path)
    return inner_train_ids, val_ids, splitter_name, cache_path

def fit_standardiser(train_mat: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    train_mat = np.asarray(train_mat, dtype=np.float32)
    if train_mat.ndim != 2:
        raise ValueError(f"fit_standardiser expects 2D input, got shape {train_mat.shape}")
    n_features = train_mat.shape[1]
    mean = np.zeros(n_features, dtype=np.float32)
    std = np.ones(n_features, dtype=np.float32)
    if train_mat.size == 0 or n_features == 0:
        return mean, std
    finite_any = np.isfinite(train_mat).any(axis=0)
    if not finite_any.any():
        return mean, std
    mean_valid = np.nanmean(train_mat[:, finite_any], axis=0)
    std_valid = np.nanstd(train_mat[:, finite_any], axis=0)
    mean_valid = np.where(np.isnan(mean_valid), 0.0, mean_valid)
    std_valid = np.where((std_valid <= 1e-8) | np.isnan(std_valid), 1.0, std_valid)
    mean[finite_any] = mean_valid.astype(np.float32)
    std[finite_any] = std_valid.astype(np.float32)
    return mean, std

def transform_standardised(mat: np.ndarray, mean: np.ndarray, std: np.ndarray) -> np.ndarray:
    out = (mat - mean) / std
    return out.astype(np.float32)

def mask_from_values(mat: np.ndarray) -> np.ndarray:
    return (~np.isnan(mat)).astype(np.float32)

def fit_target_standardiser(y_train: np.ndarray) -> Tuple[float, float]:
    y_train = np.asarray(y_train, dtype=np.float32)
    y_mean = float(np.nanmean(y_train))
    y_std = float(np.nanstd(y_train))
    if not np.isfinite(y_mean):
        y_mean = 0.0
    if not np.isfinite(y_std) or y_std < 1e-8:
        y_std = 1.0
    return y_mean, y_std

ALL_MODALITIES = ["rna", "cnv", "mut", "prot"]
CONTINUOUS_MODALITIES = {"rna", "cnv", "prot"}
CHANNEL_OFFSET = {"rna": 0, "cnv": 2, "mut": 4, "prot": 6}

def make_feature_set_channel_mask(feature_set: str) -> np.ndarray:
    mask = np.zeros(8, dtype=np.float32)
    for modality in FEATURE_SET_MODALITIES[feature_set]:
        off = CHANNEL_OFFSET[modality]
        mask[off:off + 2] = 1.0
    return mask

FEATURE_SET_CHANNEL_MASKS = {fs: make_feature_set_channel_mask(fs) for fs in FEATURE_SETS}


In [10]:

# Dataset and improved STRING-GAT model

class StringGraphDataset(torch.utils.data.Dataset):
    def __init__(
        self,
        cell_ids: List[str],
        x_tensor: np.ndarray,
        y_values: np.ndarray,
        edge_index: torch.Tensor,
        edge_attr: torch.Tensor,
    ):
        self.cell_ids = list(cell_ids)
        self.x_tensor = x_tensor.astype(np.float32, copy=False)
        self.y_values = np.asarray(y_values, dtype=np.float32)
        self.edge_index = edge_index
        self.edge_attr = edge_attr

    def __len__(self):
        return len(self.cell_ids)

    def __getitem__(self, idx):
        x = torch.from_numpy(self.x_tensor[idx])
        y = torch.tensor([self.y_values[idx]], dtype=torch.float32)

        # Active-node mask prevents global pooling over thousands of irrelevant zero/padded nodes.
        # Channel layout: value/mask for RNA 0/1, CNV 2/3, mutation 4/5, proteomics 6/7.
        continuous_or_prot_present = x[:, [1, 3, 7]].sum(dim=1) > 0
        mutated = x[:, 4].abs() > 0
        node_active = (continuous_or_prot_present | mutated).float()

        data = Data(
            x=x,
            edge_index=self.edge_index,
            edge_attr=self.edge_attr,
            y=y,
        )
        data.node_active = node_active
        data.sample_idx = torch.tensor([idx], dtype=torch.long)
        return data

class StringGATRegressor(nn.Module):
    def __init__(self, in_channels: int, hidden_channels: int = 32, heads_first: int = 4,
                 dropout: float = 0.2, use_edge_attr: bool = True):
        super().__init__()
        edge_dim = 1 if use_edge_attr else None
        self.conv1 = GATv2Conv(
            in_channels=in_channels,
            out_channels=hidden_channels,
            heads=heads_first,
            concat=True,
            dropout=dropout,
            edge_dim=edge_dim,
            add_self_loops=True,
        )
        self.norm1 = nn.LayerNorm(hidden_channels * heads_first)

        self.conv2 = GATv2Conv(
            in_channels=hidden_channels * heads_first,
            out_channels=hidden_channels,
            heads=1,
            concat=False,
            dropout=dropout,
            edge_dim=edge_dim,
            add_self_loops=True,
        )
        self.norm2 = nn.LayerNorm(hidden_channels)

        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, 1),
        )

    def encode_nodes(self, data: Data) -> torch.Tensor:
        edge_attr = data.edge_attr if hasattr(data, "edge_attr") else None
        h = self.conv1(data.x, data.edge_index, edge_attr=edge_attr)
        h = self.norm1(h)
        h = F.elu(h)
        h = self.dropout(h)

        h = self.conv2(h, data.edge_index, edge_attr=edge_attr)
        h = self.norm2(h)
        h = F.elu(h)
        h = self.dropout(h)
        return h

    def forward(self, data: Data) -> torch.Tensor:
        h = self.encode_nodes(data)

        if hasattr(data, "node_active"):
            active = data.node_active.view(-1, 1).to(h.device)
        else:
            active = torch.ones((h.shape[0], 1), dtype=h.dtype, device=h.device)

        pooled_sum = global_add_pool(h * active, data.batch)
        pooled_count = global_add_pool(active, data.batch).clamp_min(1.0)
        graph_emb = pooled_sum / pooled_count

        pred = self.head(graph_emb).view(-1)
        return pred


In [11]:

# Fold-safe node-feature construction and cached tensors

def get_arm_cohort(arm_name: str) -> List[str]:
    return list(prot_arm_data[arm_name]["eligible_cells"])

def get_drug_target_series(drug_id: str, eligible_cells: List[str]) -> pd.Series:
    sub = prism_auc[prism_auc["compound_id"] == str(drug_id)].copy()
    sub = sub[sub["depmap_id"].isin(eligible_cells)]
    if sub.empty:
        return pd.Series(dtype=float)
    y = sub.groupby("depmap_id")["y"].mean().dropna()
    return y

def modality_gene_frame_for_arm(modality: str, arm_name: str) -> pd.DataFrame:
    if modality == "rna":
        return rna_gene
    if modality == "cnv":
        return cnv_gene
    if modality == "mut":
        return mut_gene
    if modality == "prot":
        return prot_gene_by_arm[arm_name]
    raise KeyError(modality)

def build_fold_tensor_base(
    arm_name: str,
    all_ids: List[str],
    train_ids: List[str],
    cache_key: str,
) -> np.ndarray:
    cache_dir = OUT_CACHE / "fold_tensors_base"
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_path = cache_dir / f"{cache_key}.npz"
    meta_path = cache_dir / f"{cache_key}.json"

    if USE_FOLD_TENSOR_CACHE and cache_path.exists() and meta_path.exists() and not FORCE_RERUN:
        meta = load_json_if_exists(meta_path, default={}) or {}
        if meta.get("cache_version") == CACHE_VERSION:
            loaded = np.load(cache_path)
            return loaded["x"]

    n_cells = len(all_ids)
    n_nodes = len(graph_nodes)
    n_channels = 8
    x = np.zeros((n_cells, n_nodes, n_channels), dtype=np.float32)

    all_ids = list(map(str, all_ids))
    train_ids = list(map(str, train_ids))

    for modality in ALL_MODALITIES:
        frame = modality_gene_frame_for_arm(modality, arm_name).reindex(index=all_ids)
        arr = frame.to_numpy(dtype=np.float32, copy=True)

        # Notebook 13 fix: mutation mask means mutated gene, not merely observed zero after fillna(0).
        if modality == "mut":
            mask = ((arr != 0) & np.isfinite(arr)).astype(np.float32)
        else:
            mask = mask_from_values(arr)

        if modality in CONTINUOUS_MODALITIES:
            train_arr = modality_gene_frame_for_arm(modality, arm_name).reindex(index=train_ids).to_numpy(dtype=np.float32, copy=True)
            mean, std = fit_standardiser(train_arr)
            arr = transform_standardised(arr, mean, std)

        arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)

        off = CHANNEL_OFFSET[modality]
        x[:, :, off] = arr
        x[:, :, off + 1] = mask.astype(np.float32)

    np.savez_compressed(cache_path, x=x)
    write_cache_metadata(meta_path, {
        "cache_version": CACHE_VERSION,
        "arm": arm_name,
        "n_cells": int(n_cells),
        "n_nodes": int(n_nodes),
        "n_channels": int(n_channels),
        "train_ids_sha1": stable_hash(train_ids, n=24),
        "all_ids_sha1": stable_hash(all_ids, n=24),
        "graph_nodes_sha1": stable_hash(graph_nodes, n=24),
        "mutation_mask": "arr != 0",
    })
    return x

def apply_feature_set_mask(x_base: np.ndarray, feature_set: str) -> np.ndarray:
    mask = FEATURE_SET_CHANNEL_MASKS[feature_set]
    return (x_base * mask[None, None, :]).astype(np.float32, copy=False)


In [12]:

# Training and evaluation helpers

def make_loss(name: str):
    if name == "smooth_l1":
        return nn.SmoothL1Loss()
    if name == "mse":
        return nn.MSELoss()
    raise KeyError(name)

def train_one_epoch(model, loader, optimiser, criterion, device):
    model.train()
    losses = []
    for batch in loader:
        batch = batch.to(device)
        optimiser.zero_grad(set_to_none=True)
        pred = model(batch)
        loss = criterion(pred, batch.y.view(-1))
        loss.backward()
        if MODEL_CONFIG["grad_clip_norm"] is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), MODEL_CONFIG["grad_clip_norm"])
        optimiser.step()
        losses.append(float(loss.detach().cpu()))
    return float(np.mean(losses)) if losses else np.nan

@torch.no_grad()
def predict_loader(model, loader, device):
    model.eval()
    preds = []
    ys = []
    sample_indices = []
    for batch in loader:
        batch = batch.to(device)
        pred = model(batch)
        preds.append(pred.detach().cpu().numpy())
        ys.append(batch.y.view(-1).detach().cpu().numpy())
        sample_indices.append(batch.sample_idx.view(-1).detach().cpu().numpy())
    if not preds:
        return np.array([]), np.array([]), np.array([])
    return np.concatenate(preds), np.concatenate(ys), np.concatenate(sample_indices)

def fit_string_gat(
    train_dataset: StringGraphDataset,
    val_dataset: StringGraphDataset,
    seed: int,
    run_key: str,
    in_channels: int = 8,
    batch_size: Optional[int] = None,
):
    set_all_seeds(seed)

    model_dir = OUT_CACHE / "models"
    history_dir = OUT_CACHE / "histories"
    train_state_dir = OUT_CACHE / "train_states"
    fit_meta_dir = OUT_CACHE / "fit_meta"
    for d in [model_dir, history_dir, train_state_dir, fit_meta_dir]:
        d.mkdir(parents=True, exist_ok=True)

    model_path = model_dir / f"{run_key}.pt"
    history_path = history_dir / f"{run_key}.csv"
    train_state_path = train_state_dir / f"{run_key}.pt"
    fit_meta_path = fit_meta_dir / f"{run_key}.json"

    model = StringGATRegressor(
        in_channels=in_channels,
        hidden_channels=MODEL_CONFIG["hidden_channels"],
        heads_first=MODEL_CONFIG["heads_first"],
        dropout=MODEL_CONFIG["dropout"],
        use_edge_attr=MODEL_CONFIG["use_edge_attr"],
    ).to(DEVICE)

    optimiser = torch.optim.Adam(
        model.parameters(),
        lr=MODEL_CONFIG["lr"],
        weight_decay=MODEL_CONFIG["weight_decay"],
    )
    criterion = make_loss(MODEL_CONFIG["loss"])

    effective_batch_size = int(batch_size or MODEL_CONFIG["batch_size"])

    train_loader = DataLoader(
        train_dataset,
        batch_size=effective_batch_size,
        shuffle=True,
        num_workers=MODEL_CONFIG["num_workers"],
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=effective_batch_size,
        shuffle=False,
        num_workers=MODEL_CONFIG["num_workers"],
    )

    if model_path.exists() and history_path.exists() and fit_meta_path.exists() and not FORCE_RERUN:
        state = torch.load(model_path, map_location=DEVICE)
        model.load_state_dict(state)
        history_df = pd.read_csv(history_path)
        fit_info = load_json_if_exists(fit_meta_path, default={}) or {}
        fit_info.update({
            "resumed_from_epoch_checkpoint": False,
            "model_path": str(model_path),
            "history_path": str(history_path),
            "train_state_path": str(train_state_path),
            "fit_meta_path": str(fit_meta_path),
        })
        return model, history_df, fit_info

    best_state = None
    best_epoch = -1
    best_val_spearman = -np.inf
    patience_counter = 0
    history = []
    start_epoch = 1
    resumed_from_epoch_checkpoint = False

    if USE_MODEL_STATE_RESUME and train_state_path.exists() and not FORCE_RERUN:
        ckpt = torch.load(train_state_path, map_location=DEVICE)
        model.load_state_dict(ckpt["model_state_dict"])
        optimiser.load_state_dict(ckpt["optimizer_state_dict"])
        best_state = ckpt.get("best_state_dict")
        best_epoch = int(ckpt.get("best_epoch", -1))
        best_val_spearman = float(ckpt.get("best_val_spearman", -np.inf))
        patience_counter = int(ckpt.get("patience_counter", 0))
        history = list(ckpt.get("history", []))
        start_epoch = int(ckpt.get("epoch", 0)) + 1
        resumed_from_epoch_checkpoint = True

    for epoch in range(start_epoch, MODEL_CONFIG["max_epochs"] + 1):
        train_loss = train_one_epoch(model, train_loader, optimiser, criterion, DEVICE)
        val_pred, val_y, _ = predict_loader(model, val_loader, DEVICE)
        val_s = spearman_safe(val_y, val_pred)
        val_r2 = r2_safe(val_y, val_pred)
        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_spearman": val_s,
            "val_r2": val_r2,
        })

        monitor = -np.inf if np.isnan(val_s) else val_s
        if monitor > best_val_spearman + MODEL_CONFIG["min_delta"]:
            best_val_spearman = monitor
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if TRAIN_STATE_SAVE_EVERY_EPOCHS and (epoch % TRAIN_STATE_SAVE_EVERY_EPOCHS == 0):
            torch.save({
                "cache_version": CACHE_VERSION,
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimiser.state_dict(),
                "best_state_dict": best_state,
                "best_epoch": best_epoch,
                "best_val_spearman": best_val_spearman,
                "patience_counter": patience_counter,
                "history": history,
                "seed": int(seed),
                "run_key": run_key,
            }, train_state_path)

        if patience_counter >= MODEL_CONFIG["patience"]:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    history_df = pd.DataFrame(history)
    fit_info = {
        "best_epoch": int(best_epoch),
        "best_val_spearman": None if best_val_spearman == -np.inf else float(best_val_spearman),
        "n_epochs_run": int(len(history)),
        "resumed_from_epoch_checkpoint": bool(resumed_from_epoch_checkpoint),
        "model_path": str(model_path),
        "history_path": str(history_path),
        "train_state_path": str(train_state_path),
        "fit_meta_path": str(fit_meta_path),
    }

    atomic_to_csv(history_df, history_path, index=False)
    torch.save(model.state_dict(), model_path)
    write_cache_metadata(fit_meta_path, {
        "cache_version": CACHE_VERSION,
        **fit_info,
        "seed": int(seed),
        "run_key": run_key,
    })

    return model, history_df, fit_info


In [ ]:

# Checkpoint/recovery helpers and benchmark runner

def recover_rows_from_prediction_files() -> List[dict]:
    pred_paths = sorted(OUT_RESULTS.glob("predictions__seed*__*.csv"))
    if not pred_paths:
        return []

    arm_slug_to_name = {safe_slug(x): x for x in ARM_THRESHOLDS.keys()}
    feature_slug_to_name = {safe_slug(fs.replace("+", "_")): fs for fs in FEATURE_SETS}
    drug_slug_to_name = {safe_slug(drug): drug for drug in TOP10_DRUGS}

    recovered_rows = []
    for pred_path in pred_paths:
        run_key = pred_path.stem.replace("predictions__", "", 1)
        parts = run_key.split("__")
        if len(parts) != 5:
            continue

        seed_part, arm_slug, feature_slug, drug_slug, fold_part = parts
        if not seed_part.startswith("seed") or not fold_part.startswith("fold"):
            continue

        try:
            seed = int(seed_part.replace("seed", "", 1))
            fold_idx = int(fold_part.replace("fold", "", 1))
        except Exception:
            continue

        arm_name = arm_slug_to_name.get(arm_slug)
        feature_set = feature_slug_to_name.get(feature_slug)
        compound_id = drug_slug_to_name.get(drug_slug)
        if arm_name is None or feature_set is None or compound_id is None:
            continue

        model_path = OUT_CACHE / "models" / f"{run_key}.pt"
        history_path = OUT_CACHE / "histories" / f"{run_key}.csv"
        train_state_path = OUT_CACHE / "train_states" / f"{run_key}.pt"
        fit_meta_path = OUT_CACHE / "fit_meta" / f"{run_key}.json"

        if not model_path.exists():
            continue
        if not history_path.exists() and not fit_meta_path.exists():
            continue

        try:
            pred_df = pd.read_csv(pred_path)
        except Exception:
            continue
        if pred_df.empty or not {"y_true", "y_pred"}.issubset(pred_df.columns):
            continue

        y_true = pd.to_numeric(pred_df["y_true"], errors="coerce").to_numpy(dtype=float)
        y_pred = pd.to_numeric(pred_df["y_pred"], errors="coerce").to_numpy(dtype=float)

        best_epoch = np.nan
        best_val_spearman = np.nan
        n_epochs_run = np.nan
        resumed_from_epoch_checkpoint = False
        y_mean = np.nan
        y_std = np.nan

        if fit_meta_path.exists():
            meta = load_json_if_exists(fit_meta_path, default={}) or {}
            best_epoch = meta.get("best_epoch", np.nan)
            best_val_spearman = meta.get("best_val_spearman", np.nan)
            n_epochs_run = meta.get("n_epochs_run", np.nan)
            resumed_from_epoch_checkpoint = bool(meta.get("resumed_from_epoch_checkpoint", False))
            y_mean = meta.get("target_y_mean", np.nan)
            y_std = meta.get("target_y_std", np.nan)
        elif history_path.exists():
            try:
                hist_df = pd.read_csv(history_path)
                n_epochs_run = int(hist_df.shape[0])
                if not hist_df.empty and "val_spearman" in hist_df.columns and hist_df["val_spearman"].notna().any():
                    best_idx = hist_df["val_spearman"].astype(float).idxmax()
                    best_row = hist_df.loc[best_idx]
                    best_epoch = int(best_row["epoch"])
                    best_val_spearman = float(best_row["val_spearman"])
            except Exception:
                pass

        recovered_rows.append({
            "seed": seed,
            "arm": arm_name,
            "feature_set": feature_set,
            "compound_id": compound_id,
            "fold_idx": fold_idx,
            "run_key": run_key,
            "n_train": np.nan,
            "n_val": np.nan,
            "n_test": int(pred_df.shape[0]),
            "outer_splitter": None,
            "val_splitter": None,
            "spearman": spearman_safe(y_true, y_pred),
            "r2": r2_safe(y_true, y_pred),
            "best_epoch": best_epoch,
            "best_val_spearman": best_val_spearman,
            "n_epochs_run": n_epochs_run,
            "resumed_from_epoch_checkpoint": resumed_from_epoch_checkpoint,
            "threshold": ARM_THRESHOLDS[arm_name],
            "graph_n_nodes": len(graph_nodes),
            "graph_n_edges_directed": int(edge_index.shape[1]),
            "target_y_mean": y_mean,
            "target_y_std": y_std,
            "outer_split_cache": None,
            "inner_split_cache": None,
            "fold_tensor_cache_key": None,
            "predictions_path": str(pred_path),
            "model_path": str(model_path) if model_path.exists() else None,
            "history_path": str(history_path) if history_path.exists() else None,
            "train_state_path": str(train_state_path) if train_state_path.exists() else None,
            "fit_meta_path": str(fit_meta_path) if fit_meta_path.exists() else None,
        })

    return recovered_rows

checkpoint_path = OUT_RESULTS / "string_gat_checkpoint.parquet"
run_state_path = OUT_META / "run_state.json"
per_fold_rows = []

print("PROJECT_ROOT:", PROJECT_ROOT.resolve())
print("checkpoint_path:", checkpoint_path.resolve())
print("checkpoint exists:", checkpoint_path.exists())

resume_sources = []
for candidate in [OUT_RESULTS / "string_gat_checkpoint.parquet", OUT_RESULTS / "per_fold_results.parquet"]:
    if candidate.exists() and RESUME_FROM_CHECKPOINT and not FORCE_RERUN:
        try:
            tmp = pd.read_parquet(candidate)
            if not tmp.empty:
                tmp["__resume_source"] = str(candidate)
                resume_sources.append(tmp)
        except Exception:
            pass

recovered_prediction_rows = recover_rows_from_prediction_files()
if recovered_prediction_rows:
    tmp = pd.DataFrame(recovered_prediction_rows)
    tmp["__resume_source"] = "prediction_file_recovery"
    resume_sources.append(tmp)
    print(f"Recovered {len(tmp)} rows from prediction/model/history files")

if resume_sources:
    merged_resume_df = pd.concat(resume_sources, ignore_index=True)
    key_cols = ["seed", "arm", "feature_set", "compound_id", "fold_idx"]
    merged_resume_df = merged_resume_df.drop_duplicates(subset=key_cols, keep="last").reset_index(drop=True)
    per_fold_rows = merged_resume_df.drop(columns="__resume_source", errors="ignore").to_dict("records")
    print(f"Loaded existing resume rows: {len(per_fold_rows)} from {len(resume_sources)} source(s)")
else:
    print("No Notebook 13 checkpoint/results found; starting as a first-time run.")

completed_keys = {
    (r["seed"], r["arm"], r["feature_set"], r["compound_id"], r["fold_idx"])
    for r in per_fold_rows
} if per_fold_rows else set()

print("Loaded per_fold_rows:", len(per_fold_rows))
print("Loaded completed_keys:", len(completed_keys))
if per_fold_rows:
    tmp_resume_df = pd.DataFrame(per_fold_rows)
    print("Resume rows by seed:")
    print(tmp_resume_df.groupby("seed").size().sort_index())

def save_checkpoint(rows: List[dict], last_completed: Optional[dict] = None):
    if not rows:
        return
    df = pd.DataFrame(rows)
    key_cols = ["seed", "arm", "feature_set", "compound_id", "fold_idx"]
    df = df.drop_duplicates(subset=key_cols, keep="last").reset_index(drop=True)
    atomic_to_parquet(df, checkpoint_path)
    update_run_state(run_state_path, {
        "cache_version": CACHE_VERSION,
        "resume_from_checkpoint": bool(RESUME_FROM_CHECKPOINT),
        "force_rerun": bool(FORCE_RERUN),
        "n_completed_rows": int(df.shape[0]),
        "n_unique_completed_keys": int(df[key_cols].drop_duplicates().shape[0]),
        "last_completed": last_completed,
        "checkpoint_path": str(checkpoint_path),
    })

run_arms = list(ARM_THRESHOLDS.keys())
run_features = list(FEATURE_SETS)
run_drugs = list(TOP10_DRUGS)

if FAST_DEV_RUN:
    if FAST_DEV_LIMIT_ARMS is not None:
        run_arms = run_arms[:FAST_DEV_LIMIT_ARMS]
    if FAST_DEV_LIMIT_FEATURES is not None:
        run_features = run_features[:FAST_DEV_LIMIT_FEATURES]
    if FAST_DEV_LIMIT_DRUGS is not None:
        run_drugs = run_drugs[:FAST_DEV_LIMIT_DRUGS]

rows_since_last_checkpoint = 0

for seed in SEEDS:
    set_all_seeds(seed)
    print(f"\nSeed {seed}")

    for arm_name in run_arms:
        eligible_cells = get_arm_cohort(arm_name)

        completed_rows_seed_arm = sum(1 for r in per_fold_rows if r["seed"] == seed and r["arm"] == arm_name)
        if RESUME_FROM_CHECKPOINT and completed_rows_seed_arm > 0 and not FORCE_RERUN:
            print(
                f"  [resume] seed={seed} arm={arm_name}: loaded "
                f"{completed_rows_seed_arm} completed rows, scanning for missing rows only."
            )

        if len(eligible_cells) == 0:
            print(f"  {arm_name}: eligible_cells=0, skipping arm")
            continue

        group_series = cell_index.loc[eligible_cells, group_col].astype(str)
        splitter_preview = preview_outer_splitter(group_series, len(eligible_cells))
        print(f"  {arm_name}: {splitter_preview}, eligible_cells={len(eligible_cells)}")

        arm_drug_cache = {}
        for compound_id in run_drugs:
            y_series = get_drug_target_series(compound_id, eligible_cells)
            y_series = y_series[y_series.index.isin(eligible_cells)]

            if y_series.shape[0] < MIN_CELLS_PER_DRUG:
                arm_drug_cache[compound_id] = {"skip": True, "y_series": y_series}
                continue

            sample_ids = y_series.index.tolist()
            sample_groups = group_series.loc[sample_ids]
            outer_folds, splitter_name, outer_split_cache = get_outer_folds_cached(
                sample_ids, sample_groups, seed, arm_name, compound_id
            )

            arm_drug_cache[compound_id] = {
                "skip": False,
                "y_series": y_series,
                "sample_ids": sample_ids,
                "sample_groups": sample_groups,
                "outer_folds": outer_folds,
                "splitter_name": splitter_name,
                "outer_split_cache": outer_split_cache,
            }

        drugs_run = 0
        drugs_skipped = 0
        last_completed_arm = None

        for drug_idx, compound_id in enumerate(run_drugs, start=1):
            cache_entry = arm_drug_cache[compound_id]
            if cache_entry["skip"]:
                drugs_skipped += 1
                continue

            y_series = cache_entry["y_series"]
            sample_groups = cache_entry["sample_groups"]
            outer_folds = cache_entry["outer_folds"]
            splitter_name = cache_entry["splitter_name"]
            outer_split_cache = cache_entry["outer_split_cache"]

            drugs_run += 1
            print(
                f"    [{arm_name}] ({drug_idx}/{len(run_drugs)}) "
                f"drug={compound_id} n_cells={y_series.shape[0]}",
                flush=True,
            )

            last_completed_for_drug = None

            for fold_idx, (train_ids, test_ids) in enumerate(outer_folds):
                pending_feature_sets = [
                    fs for fs in run_features
                    if (seed, arm_name, fs, compound_id, fold_idx) not in completed_keys
                ]
                if not pending_feature_sets:
                    continue

                inner_train_ids, val_ids, val_splitter, inner_split_cache = get_train_val_split_cached(
                    train_ids=train_ids,
                    groups=sample_groups,
                    seed=seed + fold_idx,
                    arm_name=arm_name,
                    compound_id=compound_id,
                    fold_idx=fold_idx,
                )

                fold_tensor_payload_base = {
                    "seed": seed,
                    "arm": arm_name,
                    "drug": compound_id,
                    "fold_idx": fold_idx,
                    "train_ids": train_ids,
                    "test_ids": test_ids,
                    "cache_version": CACHE_VERSION,
                }
                fold_cache_key_base = stable_hash(fold_tensor_payload_base, n=24)

                all_ids = train_ids + test_ids
                x_base = build_fold_tensor_base(
                    arm_name=arm_name,
                    all_ids=all_ids,
                    train_ids=train_ids,
                    cache_key=fold_cache_key_base,
                )
                id_to_pos = {cid: i for i, cid in enumerate(all_ids)}

                train_pos = [id_to_pos[cid] for cid in inner_train_ids]
                val_pos = [id_to_pos[cid] for cid in val_ids]
                test_pos = [id_to_pos[cid] for cid in test_ids]

                y_train = y_series.loc[inner_train_ids].to_numpy(dtype=np.float32)
                y_val = y_series.loc[val_ids].to_numpy(dtype=np.float32)
                y_test = y_series.loc[test_ids].to_numpy(dtype=np.float32)

                # Notebook 13 fix: target is standardised only within the inner training fold.
                y_mean, y_std = fit_target_standardiser(y_train)
                y_train_model = ((y_train - y_mean) / y_std).astype(np.float32)
                y_val_model = ((y_val - y_mean) / y_std).astype(np.float32)
                y_test_model = ((y_test - y_mean) / y_std).astype(np.float32)

                effective_batch_size = get_batch_size_for_arm(arm_name)

                for feature_set in pending_feature_sets:
                    x_all = apply_feature_set_mask(x_base, feature_set)

                    train_ds = StringGraphDataset(inner_train_ids, x_all[train_pos], y_train_model, edge_index=edge_index, edge_attr=edge_attr)
                    val_ds = StringGraphDataset(val_ids, x_all[val_pos], y_val_model, edge_index=edge_index, edge_attr=edge_attr)
                    test_ds = StringGraphDataset(test_ids, x_all[test_pos], y_test_model, edge_index=edge_index, edge_attr=edge_attr)

                    run_key = "__".join([
                        f"seed{seed}",
                        safe_slug(arm_name),
                        safe_slug(feature_set.replace("+", "_")),
                        safe_slug(compound_id),
                        f"fold{fold_idx}",
                    ])

                    model, history_df, fit_info = fit_string_gat(
                        train_ds,
                        val_ds,
                        seed=seed + fold_idx,
                        run_key=run_key,
                        in_channels=8,
                        batch_size=effective_batch_size,
                    )

                    # Save target scaler metadata alongside model metadata.
                    fit_meta_path = Path(fit_info.get("fit_meta_path"))
                    fit_meta = load_json_if_exists(fit_meta_path, default={}) or {}
                    fit_meta.update({
                        "target_y_mean": y_mean,
                        "target_y_std": y_std,
                        "target_standardisation": "fold_safe_inner_train_only",
                    })
                    write_cache_metadata(fit_meta_path, fit_meta)

                    test_loader = DataLoader(test_ds, batch_size=effective_batch_size, shuffle=False, num_workers=0)
                    test_pred_scaled, test_y_scaled, _ = predict_loader(model, test_loader, DEVICE)

                    # Notebook 13 fix: metrics and saved predictions are back on original AUC scale.
                    test_pred = (test_pred_scaled * y_std) + y_mean
                    test_y_out = y_test.astype(np.float32)

                    pred_df = pd.DataFrame({
                        "depmap_id": test_ids,
                        "y_true": test_y_out,
                        "y_pred": test_pred,
                        "y_pred_scaled": test_pred_scaled,
                        "y_true_scaled": y_test_model,
                        "target_y_mean": y_mean,
                        "target_y_std": y_std,
                        "seed": seed,
                        "arm": arm_name,
                        "feature_set": feature_set,
                        "compound_id": compound_id,
                        "fold_idx": fold_idx,
                    })
                    pred_path = OUT_RESULTS / f"predictions__{run_key}.csv"
                    atomic_to_csv(pred_df, pred_path, index=False)

                    row = {
                        "seed": seed,
                        "arm": arm_name,
                        "feature_set": feature_set,
                        "compound_id": compound_id,
                        "fold_idx": fold_idx,
                        "run_key": run_key,
                        "n_train": len(inner_train_ids),
                        "n_val": len(val_ids),
                        "n_test": len(test_ids),
                        "outer_splitter": splitter_name,
                        "val_splitter": val_splitter,
                        "spearman": spearman_safe(test_y_out, test_pred),
                        "r2": r2_safe(test_y_out, test_pred),
                        "best_epoch": fit_info["best_epoch"],
                        "best_val_spearman": fit_info["best_val_spearman"],
                        "n_epochs_run": fit_info["n_epochs_run"],
                        "resumed_from_epoch_checkpoint": fit_info.get("resumed_from_epoch_checkpoint", False),
                        "threshold": ARM_THRESHOLDS[arm_name],
                        "graph_n_nodes": len(graph_nodes),
                        "graph_n_edges_directed": int(edge_index.shape[1]),
                        "target_y_mean": y_mean,
                        "target_y_std": y_std,
                        "outer_split_cache": str(outer_split_cache),
                        "inner_split_cache": str(inner_split_cache),
                        "fold_tensor_cache_key": fold_cache_key_base,
                        "predictions_path": str(pred_path),
                        "model_path": fit_info.get("model_path"),
                        "history_path": fit_info.get("history_path"),
                        "train_state_path": fit_info.get("train_state_path"),
                        "fit_meta_path": fit_info.get("fit_meta_path"),
                    }
                    per_fold_rows.append(row)
                    completed_keys.add((seed, arm_name, feature_set, compound_id, fold_idx))
                    rows_since_last_checkpoint += 1

                    last_completed_for_drug = {
                        "seed": seed,
                        "arm": arm_name,
                        "feature_set": feature_set,
                        "compound_id": compound_id,
                        "fold_idx": fold_idx,
                    }
                    last_completed_arm = last_completed_for_drug

                    print(
                        f"        finished feature_set={feature_set} | "
                        f"drug {drug_idx}/{len(run_drugs)} | "
                        f"fold {fold_idx + 1}/{len(outer_folds)} | "
                        f"spearman={row['spearman']:.4f} | pred_std={np.nanstd(test_pred):.5f}",
                        flush=True,
                    )

                    if CHECKPOINT_EVERY_N_ROWS and rows_since_last_checkpoint >= CHECKPOINT_EVERY_N_ROWS:
                        save_checkpoint(per_fold_rows, last_completed=last_completed_for_drug)
                        rows_since_last_checkpoint = 0

                    del model, train_ds, val_ds, test_ds, test_loader, x_all
                    gc.collect()
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()

                del x_base
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            if CHECKPOINT_EVERY_N_DRUGS and (drug_idx % CHECKPOINT_EVERY_N_DRUGS == 0 or drug_idx == len(run_drugs)):
                save_checkpoint(per_fold_rows, last_completed=last_completed_for_drug)
                print(f"    [mid-checkpoint] drug {drug_idx}/{len(run_drugs)}", flush=True)

        print(f"    [{arm_name}] drugs_run={drugs_run}, drugs_skipped={drugs_skipped}")
        save_checkpoint(per_fold_rows, last_completed=last_completed_arm)
        print("    [checkpoint] saved")

save_checkpoint(per_fold_rows)
print("Benchmark loop complete. Rows:", len(per_fold_rows))


PROJECT_ROOT: /home/andrija/Desktop/Final Year Project/FYP
checkpoint_path: /home/andrija/Desktop/Final Year Project/FYP/artefacts/reports/notebook 13/results/string_gat_checkpoint.parquet
checkpoint exists: False
No Notebook 13 checkpoint/results found; starting as a first-time run.
Loaded per_fold_rows: 0
Loaded completed_keys: 0

Seed 19537
  prot_procan_depmapSanger: GroupKFold(n_splits=10), eligible_cells=485
    [prot_procan_depmapSanger] (1/5) drug=IXAZOMIB (BRD:BRD-K78659596-001-03-9) n_cells=272
        finished feature_set=rna | drug 1/5 | fold 1/10 | spearman=-0.2003 | pred_std=0.00302
        finished feature_set=cnv | drug 1/5 | fold 1/10 | spearman=0.1438 | pred_std=0.00101
        finished feature_set=mut | drug 1/5 | fold 1/10 | spearman=0.2940 | pred_std=0.00015
        finished feature_set=prot | drug 1/5 | fold 1/10 | spearman=0.1221 | pred_std=0.00483
        finished feature_set=rna+cnv | drug 1/5 | fold 1/10 | spearman=-0.0745 | pred_std=0.00207


KeyboardInterrupt: 

: 

In [ ]:

# Aggregate benchmark outputs

if not per_fold_rows and checkpoint_path.exists():
    per_fold_df = pd.read_parquet(checkpoint_path)
else:
    per_fold_df = pd.DataFrame(per_fold_rows)

if per_fold_df.empty:
    raise RuntimeError("No benchmark rows were produced. Check paths, cohort sizes, and minimum-cell settings.")

key_cols = ["seed", "arm", "feature_set", "compound_id", "fold_idx"]
per_fold_df = per_fold_df.drop_duplicates(subset=key_cols, keep="last")
per_fold_df = per_fold_df.sort_values(key_cols).reset_index(drop=True)
per_fold_df.to_csv(OUT_RESULTS / "per_fold_results.csv", index=False)
per_fold_df.to_parquet(OUT_RESULTS / "per_fold_results.parquet", index=False)

per_drug_seed = (
    per_fold_df.groupby(["seed", "arm", "feature_set", "compound_id"], as_index=False)
    .agg(
        mean_spearman=("spearman", "mean"),
        median_spearman=("spearman", "median"),
        mean_r2=("r2", "mean"),
        median_r2=("r2", "median"),
        n_folds=("fold_idx", "nunique"),
    )
)
per_drug_seed.to_csv(OUT_RESULTS / "per_drug_seed_summary.csv", index=False)

per_drug = (
    per_drug_seed.groupby(["arm", "feature_set", "compound_id"], as_index=False)
    .agg(
        mean_spearman=("mean_spearman", "mean"),
        median_spearman=("median_spearman", "median"),
        mean_r2=("mean_r2", "mean"),
        median_r2=("median_r2", "median"),
        n_seed_rows=("seed", "nunique"),
    )
)
per_drug.to_csv(OUT_RESULTS / "per_drug_summary.csv", index=False)

seed_summary = (
    per_fold_df.groupby(["seed"], as_index=False)
    .agg(
        mean_spearman=("spearman", "mean"),
        median_spearman=("spearman", "median"),
        mean_r2=("r2", "mean"),
        median_r2=("r2", "median"),
        n_rows=("fold_idx", "count"),
    )
)
seed_summary.to_csv(OUT_RESULTS / "seed_summary.csv", index=False)

arm_feature_summary = (
    per_drug_seed.groupby(["arm", "feature_set"], as_index=False)
    .agg(
        mean_spearman=("mean_spearman", "mean"),
        median_spearman=("median_spearman", "median"),
        mean_r2=("mean_r2", "mean"),
        median_r2=("median_r2", "median"),
        n_drug_seed=("compound_id", "count"),
    )
    .sort_values(["arm", "mean_spearman"], ascending=[True, False])
)
arm_feature_summary.to_csv(OUT_RESULTS / "arm_feature_summary.csv", index=False)

arm_summary = (
    per_drug_seed.groupby(["arm"], as_index=False)
    .agg(
        mean_spearman=("mean_spearman", "mean"),
        median_spearman=("median_spearman", "median"),
        mean_r2=("mean_r2", "mean"),
        median_r2=("median_r2", "median"),
        n_rows=("compound_id", "count"),
    )
    .sort_values("mean_spearman", ascending=False)
)
arm_summary.to_csv(OUT_RESULTS / "arm_summary.csv", index=False)

best_by_arm = (
    arm_feature_summary
    .sort_values(["arm", "mean_spearman"], ascending=[True, False])
    .groupby("arm", as_index=False)
    .first()
)
best_by_arm.to_csv(OUT_RESULTS / "best_feature_set_by_arm.csv", index=False)

display(seed_summary)
display(arm_summary)
display(best_by_arm)


In [ ]:

# Diagnostics: prediction variance, graph coverage, and baseline discovery

def prediction_variance_diagnostics(per_fold_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in per_fold_df.iterrows():
        pred_path = row.get("predictions_path")
        if not isinstance(pred_path, str) or not Path(pred_path).exists():
            continue
        try:
            pred_df = pd.read_csv(pred_path)
        except Exception:
            continue
        if pred_df.empty or not {"y_true", "y_pred"}.issubset(pred_df.columns):
            continue
        y = pd.to_numeric(pred_df["y_true"], errors="coerce").to_numpy(dtype=float)
        p = pd.to_numeric(pred_df["y_pred"], errors="coerce").to_numpy(dtype=float)
        y_std = float(np.nanstd(y))
        p_std = float(np.nanstd(p))
        rows.append({
            "seed": row["seed"],
            "arm": row["arm"],
            "feature_set": row["feature_set"],
            "compound_id": row["compound_id"],
            "fold_idx": row["fold_idx"],
            "spearman": row["spearman"],
            "r2": row["r2"],
            "y_mean": float(np.nanmean(y)),
            "y_std": y_std,
            "pred_mean": float(np.nanmean(p)),
            "pred_std": p_std,
            "pred_to_y_std_ratio": p_std / y_std if y_std > 1e-12 else np.nan,
            "abs_mean_shift": abs(float(np.nanmean(p)) - float(np.nanmean(y))),
        })
    return pd.DataFrame(rows)

pred_var_df = prediction_variance_diagnostics(per_fold_df)
pred_var_df.to_csv(OUT_RESULTS / "prediction_variance_diagnostics.csv", index=False)

if not pred_var_df.empty:
    pred_var_by_arm = (
        pred_var_df.groupby("arm", as_index=False)
        .agg(
            mean_spearman=("spearman", "mean"),
            mean_r2=("r2", "mean"),
            median_y_std=("y_std", "median"),
            median_pred_std=("pred_std", "median"),
            median_pred_to_y_std_ratio=("pred_to_y_std_ratio", "median"),
            median_abs_mean_shift=("abs_mean_shift", "median"),
        )
        .sort_values("mean_spearman", ascending=False)
    )
    pred_var_by_arm.to_csv(OUT_RESULTS / "prediction_variance_by_arm.csv", index=False)
    display(Markdown("## Prediction variance diagnostics by arm"))
    display(pred_var_by_arm)

    near_constant = pred_var_df.sort_values("pred_to_y_std_ratio").head(30)
    near_constant.to_csv(OUT_RESULTS / "near_constant_prediction_rows.csv", index=False)
    display(Markdown("## Rows with smallest prediction variance ratio"))
    display(near_constant)

display(Markdown("## STRING graph node diagnostics"))
print("Total graph nodes:", len(graph_nodes))
print("Directed edges:", int(edge_index.shape[1]))
display(node_meta["in_string_high_confidence_component"].value_counts(dropna=False).reset_index(name="n_nodes"))

def discover_tabular_baselines(scan_dirs: List[Path]) -> pd.DataFrame:
    records = []
    for d in scan_dirs:
        if not d.exists():
            continue
        for path in list(d.rglob("*.csv")) + list(d.rglob("*.parquet")):
            try:
                df = read_table(path)
            except Exception:
                continue
            cols_lower = {str(c).lower(): c for c in df.columns}
            colmap = {}
            for target_name, cands in {
                "arm": ["arm", "proteomics_arm"],
                "feature_set": ["feature_set", "features", "feature_combo"],
                "compound_id": ["compound_id", "drug_id", "compound"],
                "mean_spearman": ["mean_spearman", "spearman_mean", "cv_mean_spearman", "spearman"],
            }.items():
                for cand in cands:
                    if cand in cols_lower:
                        colmap[cols_lower[cand]] = target_name
                        break
            if set(colmap.values()) >= {"arm", "feature_set", "mean_spearman"}:
                tmp = df.rename(columns=colmap).copy()
                tmp["source_file"] = str(path)
                if "compound_id" not in tmp.columns:
                    tmp["compound_id"] = "__all__"
                tmp = tmp[["arm", "feature_set", "compound_id", "mean_spearman", "source_file"]].copy()
                records.append(tmp)
    if not records:
        return pd.DataFrame(columns=["arm", "feature_set", "compound_id", "mean_spearman", "source_file"])
    out = pd.concat(records, ignore_index=True)
    out["arm"] = out["arm"].astype(str)
    out["feature_set"] = out["feature_set"].astype(str).map(normalise_feature_set_name)
    out["compound_id"] = out["compound_id"].astype(str)
    out["mean_spearman"] = pd.to_numeric(out["mean_spearman"], errors="coerce")
    out = out.dropna(subset=["mean_spearman"])
    return out

baseline_df = discover_tabular_baselines(BASELINE_SCAN_DIRS)
baseline_df.to_csv(OUT_COMPARE / "discovered_tabular_baselines.csv", index=False)

graph_compare = arm_feature_summary.copy()
graph_compare["compound_id"] = "__all__"
baseline_compare = graph_compare.merge(
    baseline_df,
    on=["arm", "feature_set", "compound_id"],
    how="left",
    suffixes=("_string_gat_notebook13", "_tabular"),
)
if not baseline_compare.empty and "mean_spearman_tabular" in baseline_compare.columns:
    baseline_compare["delta_vs_tabular"] = baseline_compare["mean_spearman_string_gat_notebook13"] - baseline_compare["mean_spearman_tabular"]
baseline_compare.to_csv(OUT_COMPARE / "arm_feature_vs_tabular.csv", index=False)
display(Markdown("## Baseline comparison preview"))
display(baseline_compare.head(20))


In [ ]:

# Optional interpretability: gradient x input node importance for best arm rows

def load_saved_model_for_row(row: pd.Series) -> nn.Module:
    model = StringGATRegressor(
        in_channels=8,
        hidden_channels=MODEL_CONFIG["hidden_channels"],
        heads_first=MODEL_CONFIG["heads_first"],
        dropout=MODEL_CONFIG["dropout"],
        use_edge_attr=MODEL_CONFIG["use_edge_attr"],
    ).to(DEVICE)
    model_path = Path(row["model_path"]) if "model_path" in row and pd.notna(row["model_path"]) else (OUT_CACHE / "models" / f"{row['run_key']}.pt")
    state = torch.load(model_path, map_location=DEVICE)
    model.load_state_dict(state)
    model.eval()
    return model

def gradient_x_input_importance(model: nn.Module, data: Data) -> Tuple[np.ndarray, Dict[str, float]]:
    model.eval()
    data = copy.deepcopy(data)
    data = data.to(DEVICE)
    x = data.x.clone().detach().requires_grad_(True)
    data.x = x
    pred = model(data).sum()
    pred.backward()
    grad = x.grad.detach()
    attr = (x * grad).abs().detach().cpu().numpy()
    gene_importance = attr.sum(axis=1)
    modality_importance = {
        "rna": float(attr[:, 0:2].sum()),
        "cnv": float(attr[:, 2:4].sum()),
        "mut": float(attr[:, 4:6].sum()),
        "prot": float(attr[:, 6:8].sum()),
    }
    return gene_importance, modality_importance

def select_explainer_rows(per_drug_seed: pd.DataFrame, n_per_arm: int = 1) -> pd.DataFrame:
    picked = []
    for arm_name, sub in per_drug_seed.groupby("arm"):
        top = sub.sort_values(["mean_spearman", "median_spearman"], ascending=False).head(n_per_arm)
        picked.append(top)
    if not picked:
        return pd.DataFrame()
    return pd.concat(picked, ignore_index=True)

explainer_targets = select_explainer_rows(per_drug_seed, n_per_arm=1)
explainer_targets.to_csv(OUT_INTERP / "explainer_targets.csv", index=False)
display(explainer_targets)

explainer_rows = []
modality_rows = []

for _, target_row in explainer_targets.iterrows():
    arm_name = target_row["arm"]
    feature_set = target_row["feature_set"]
    compound_id = target_row["compound_id"]
    seed = int(target_row["seed"])

    eligible_cells = get_arm_cohort(arm_name)
    y_series = get_drug_target_series(compound_id, eligible_cells)
    sample_ids = y_series.index.tolist()
    sample_groups = cell_index.loc[sample_ids, group_col].astype(str)
    outer_folds, _, _ = get_outer_folds_cached(sample_ids, sample_groups, seed, arm_name, compound_id)

    fold_means = []
    fold_mods = []
    for fold_idx, (train_ids, test_ids) in enumerate(outer_folds):
        key_match = (
            (per_fold_df["seed"] == seed) &
            (per_fold_df["arm"] == arm_name) &
            (per_fold_df["feature_set"] == feature_set) &
            (per_fold_df["compound_id"] == compound_id) &
            (per_fold_df["fold_idx"] == fold_idx)
        )
        if key_match.sum() == 0:
            continue

        model_row = per_fold_df[key_match].iloc[0]
        y_mean = float(model_row.get("target_y_mean", 0.0))
        y_std = float(model_row.get("target_y_std", 1.0))
        if not np.isfinite(y_std) or y_std <= 0:
            y_std = 1.0

        fold_tensor_payload_base = {
            "seed": seed,
            "arm": arm_name,
            "drug": compound_id,
            "fold_idx": fold_idx,
            "train_ids": train_ids,
            "test_ids": test_ids,
            "cache_version": CACHE_VERSION,
        }
        cache_key_base = stable_hash(fold_tensor_payload_base, n=24)

        all_ids = train_ids + test_ids
        x_base = build_fold_tensor_base(arm_name=arm_name, all_ids=all_ids, train_ids=train_ids, cache_key=cache_key_base)
        x_all = apply_feature_set_mask(x_base, feature_set)
        id_to_pos = {cid: i for i, cid in enumerate(all_ids)}
        model = load_saved_model_for_row(model_row)

        explained_gene_scores = []
        explained_modality_scores = []
        for cid in test_ids[:8]:
            pos = id_to_pos[cid]
            y_scaled = (float(y_series.loc[cid]) - y_mean) / y_std
            x = torch.from_numpy(x_all[pos])
            continuous_or_prot_present = x[:, [1, 3, 7]].sum(dim=1) > 0
            mutated = x[:, 4].abs() > 0
            node_active = (continuous_or_prot_present | mutated).float()
            data = Data(
                x=x,
                edge_index=edge_index,
                edge_attr=edge_attr,
                y=torch.tensor([y_scaled], dtype=torch.float32),
            )
            data.batch = torch.zeros(data.x.shape[0], dtype=torch.long)
            data.node_active = node_active
            gene_imp, modality_imp = gradient_x_input_importance(model, data)
            explained_gene_scores.append(gene_imp)
            explained_modality_scores.append(modality_imp)

        if explained_gene_scores:
            mean_gene_imp = np.mean(np.vstack(explained_gene_scores), axis=0)
            fold_means.append(mean_gene_imp)
            mean_modality_imp = {k: float(np.mean([d[k] for d in explained_modality_scores])) for k in ["rna", "cnv", "mut", "prot"]}
            mean_modality_imp.update({
                "seed": seed,
                "arm": arm_name,
                "feature_set": feature_set,
                "compound_id": compound_id,
                "fold_idx": fold_idx,
            })
            fold_mods.append(mean_modality_imp)

        del model, x_all, x_base
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if fold_means:
        agg_gene_imp = np.mean(np.vstack(fold_means), axis=0)
        gene_rank_df = pd.DataFrame({"gene_symbol": graph_nodes, "importance": agg_gene_imp}).sort_values("importance", ascending=False).reset_index(drop=True)
        gene_path = OUT_INTERP / f"node_importance__{safe_slug(arm_name)}__{feature_set.replace('+', '_')}__{safe_slug(compound_id)}__seed{seed}.csv"
        gene_rank_df.to_csv(gene_path, index=False)
        explainer_rows.append({
            "seed": seed,
            "arm": arm_name,
            "feature_set": feature_set,
            "compound_id": compound_id,
            "node_importance_path": str(gene_path),
        })

    if fold_mods:
        mod_df = pd.DataFrame(fold_mods)
        mod_path = OUT_INTERP / f"modality_importance__{safe_slug(arm_name)}__{feature_set.replace('+', '_')}__{safe_slug(compound_id)}__seed{seed}.csv"
        mod_df.to_csv(mod_path, index=False)
        modality_rows.append(mod_df)

explainer_index = pd.DataFrame(explainer_rows)
explainer_index.to_csv(OUT_INTERP / "explainer_index.csv", index=False)

if modality_rows:
    modality_importance_df = pd.concat(modality_rows, ignore_index=True)
else:
    modality_importance_df = pd.DataFrame(columns=["seed", "arm", "feature_set", "compound_id", "fold_idx", "rna", "cnv", "mut", "prot"])

modality_importance_df.to_csv(OUT_INTERP / "modality_importance_all.csv", index=False)
display(explainer_index)
display(modality_importance_df.head())


In [ ]:

# Optional pathway enrichment from node-importance rankings

def find_first_existing_path(paths: List[Path]) -> Optional[Path]:
    for p in paths:
        if Path(p).exists():
            return Path(p)
    return None

def load_gmt(path: Path) -> Dict[str, set]:
    gene_sets = {}
    with open(path, "r") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 3:
                continue
            name = parts[0]
            genes = {canonicalise_gene_token(g) for g in parts[2:]}
            genes = {g for g in genes if g is not None}
            if genes:
                gene_sets[name] = genes
    return gene_sets

def run_ora(top_genes: List[str], universe: List[str], gene_sets: Dict[str, set]) -> pd.DataFrame:
    top_genes = [g for g in top_genes if g in universe]
    universe = list(dict.fromkeys(universe))
    universe_set = set(universe)
    top_set = set(top_genes)
    rows = []
    M = len(universe_set)
    N = len(top_set)
    for name, gs in gene_sets.items():
        gs = gs & universe_set
        if len(gs) == 0:
            continue
        k = len(top_set & gs)
        n = len(gs)
        p = hypergeom.sf(k - 1, M, n, N) if k > 0 else 1.0
        rows.append({
            "pathway": name,
            "overlap": k,
            "set_size": n,
            "query_size": N,
            "universe_size": M,
            "p_value": p,
            "leading_genes": ";".join(sorted(top_set & gs)),
        })
    if not rows:
        return pd.DataFrame(columns=["pathway", "overlap", "set_size", "query_size", "universe_size", "p_value", "fdr", "leading_genes"])
    df = pd.DataFrame(rows).sort_values(["p_value", "overlap"], ascending=[True, False]).reset_index(drop=True)
    df["fdr"] = bh_fdr(df["p_value"].to_numpy())
    return df.sort_values(["fdr", "p_value", "overlap"], ascending=[True, True, False]).reset_index(drop=True)

gmt_path = find_first_existing_path(PATHWAY_GMT_CANDIDATES)
pathway_summary_rows = []

if gmt_path is not None and not explainer_index.empty:
    gene_sets = load_gmt(gmt_path)
    universe = graph_nodes
    for _, row in explainer_index.iterrows():
        imp_df = pd.read_csv(row["node_importance_path"])
        top_genes = imp_df.head(200)["gene_symbol"].astype(str).tolist()
        out_df = run_ora(top_genes, universe, gene_sets)
        out_path = OUT_INTERP / f"ora__{safe_slug(row['arm'])}__{row['feature_set'].replace('+', '_')}__{safe_slug(row['compound_id'])}__seed{row['seed']}.csv"
        out_df.to_csv(out_path, index=False)
        pathway_summary_rows.append({
            "seed": row["seed"],
            "arm": row["arm"],
            "feature_set": row["feature_set"],
            "compound_id": row["compound_id"],
            "gmt_path": str(gmt_path),
            "ora_path": str(out_path),
            "n_fdr_lt_0_05": int((out_df["fdr"] < 0.05).sum()) if not out_df.empty else 0,
        })
else:
    skip_info = {
        "reason": "No local GMT file found" if gmt_path is None else "No explainer targets available",
        "searched_paths": [str(p) for p in PATHWAY_GMT_CANDIDATES],
    }
    with open(OUT_INTERP / "pathway_enrichment_skipped.json", "w") as f:
        json.dump(skip_info, f, indent=2)

pathway_summary_df = pd.DataFrame(pathway_summary_rows)
pathway_summary_df.to_csv(OUT_INTERP / "pathway_enrichment_summary.csv", index=False)
display(pathway_summary_df.head() if not pathway_summary_df.empty else pathway_summary_df)


In [ ]:

# Reporting figures and final lock summary

plt.figure(figsize=(6, 4))
plt.bar(seed_summary["seed"].astype(str), seed_summary["mean_spearman"])
plt.title("Notebook 13 improved STRING-GAT mean Spearman by seed")
plt.xlabel("Seed")
plt.ylabel("Mean Spearman")
plt.tight_layout()
plt.savefig(OUT_FIGS / "seed_mean_spearman.png", dpi=180, bbox_inches="tight")
plt.show()

for arm_name, sub in arm_feature_summary.groupby("arm"):
    ordered = sub.sort_values("mean_spearman", ascending=False).reset_index(drop=True)
    plt.figure(figsize=(8, max(4, 0.28 * len(ordered))))
    plt.barh(ordered["feature_set"], ordered["mean_spearman"])
    plt.gca().invert_yaxis()
    plt.title(f"Notebook 13 improved STRING-GAT | {arm_name}: mean Spearman by feature set")
    plt.xlabel("Mean Spearman")
    plt.tight_layout()
    plt.savefig(OUT_FIGS / f"{safe_slug(arm_name)}__feature_set_bar.png", dpi=180, bbox_inches="tight")
    plt.show()

best_drug_rows = per_drug.sort_values("mean_spearman", ascending=False).head(20)
plt.figure(figsize=(9, max(4, 0.30 * len(best_drug_rows))))
labels = [f"{r.arm} | {r.feature_set} | {r.compound_id}" for r in best_drug_rows.itertuples()]
plt.barh(labels, best_drug_rows["mean_spearman"])
plt.gca().invert_yaxis()
plt.title("Notebook 13 improved STRING-GAT top 20 drug-level configurations")
plt.xlabel("Mean Spearman")
plt.tight_layout()
plt.savefig(OUT_FIGS / "top20_drug_level_configurations.png", dpi=180, bbox_inches="tight")
plt.show()

if not modality_importance_df.empty:
    mod_mean = modality_importance_df.groupby(["arm", "feature_set"], as_index=False)[["rna", "cnv", "mut", "prot"]].mean()
    mod_mean.to_csv(OUT_INTERP / "modality_importance_mean_by_arm_feature.csv", index=False)
    display(mod_mean.head())

final_lock = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "target": PRIMARY_TARGET,
    "top10_drugs": TOP10_DRUGS,
    "seeds": SEEDS,
    "n_splits_desired": N_SPLITS_DESIRED,
    "min_cells_per_drug": MIN_CELLS_PER_DRUG,
    "group_column": group_col,
    "arm_thresholds": ARM_THRESHOLDS,
    "feature_sets": FEATURE_SETS,
    "string_graph": graph_meta,
    "model_config": MODEL_CONFIG,
    "cache_version": CACHE_VERSION,
    "notebook_13_changes": {
        "connected_graph_only": True,
        "active_node_masked_pooling": True,
        "mutation_mask": "mutated genes only, arr != 0",
        "target_standardisation": "fold-safe inner train mean/std; predictions unscaled before metrics",
        "checkpoint_every_n_rows": CHECKPOINT_EVERY_N_ROWS,
        "model_state_resume": USE_MODEL_STATE_RESUME,
        "split_cache": USE_SPLIT_CACHE,
        "fold_tensor_cache": USE_FOLD_TENSOR_CACHE,
    },
    "best_feature_set_by_arm": best_by_arm.to_dict("records"),
    "arm_summary": arm_summary.to_dict("records"),
    "seed_summary": seed_summary.to_dict("records"),
    "input_registry": input_registry,
}

with open(OUT_META / "string_gat_notebook13_lock.json", "w") as f:
    json.dump(final_lock, f, indent=2)

display(Markdown("## Final lock summary written to `artefacts/metadata/notebook 13/string_gat_notebook13_lock.json`"))
display(best_by_arm)
